# M5 — Pothole Segmentation
**Abdulrahman Alsalmi — 225493**

In this notebook, we will train a model to identify pothole pixels and estimate the area they cover in an image. We will prepare the data, review SAM mask proposals, train YOLO11n-seg, and evaluate its predictions against the reference masks.

We will use the Pothole Videos Dataset, which contains 619 paired RGB and mask videos: 372 for training, 124 for validation, and 123 for testing. We will keep these splits and use 12 fixed frames from each video for training and evaluation.

## 1. Required Libraries

First, we will install the libraries needed for segmentation, evaluation, visual review, and experiment tracking. We will keep Colab's existing PyTorch installation.

In [ ]:
import importlib.metadata as md
import subprocess
import sys

# Keep Colab's CUDA-compatible PyTorch installation.
required = {"ultralytics": "8.4.197", "pycocotools": "2.0.10", "ipywidgets": "8.1.7", "wandb": "0.22.3", "PyYAML": "6.0.3"}
missing = []
for package, version in required.items():
    try:
        installed = md.version(package)
    except md.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Required packages are installed. PyTorch was not explicitly replaced.")


## 2. Experiment Settings

We will set the model, image size, training duration, and frame sampling plan. We will also set a random seed and prepare functions for saving files and checking their contents.

In [ ]:
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path, PurePosixPath

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Markdown, display
from PIL import Image
from ultralytics import SAM, YOLO, settings

settings.update({"wandb": False, "mlflow": False, "comet": False})
cv2.setNumThreads(2)
PROTOCOL = {
    "version": "m5-v1", "seed": 42, "model": "yolo11n-seg.pt",
    "imgsz": 640, "epochs": 30, "batch": 8, "workers": 2,
    "optimizer": "AdamW", "lr0": 0.001, "lrf": 0.01, "weight_decay": 0.0005,
    "ultralytics": "8.4.197", "pycocotools": "2.0.10",
    "frame_indices": [0, 4, 9, 13, 17, 21, 26, 30, 34, 38, 43, 47],
    "mask_threshold": 127, "instance_rule": "8-connected foreground components; no area filtering",
    "eval_conf_floor": 0.001, "nms_iou": 0.7, "max_det": 100,
    "threshold_grid": [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90],
    "checkpoint_selection": "Ultralytics validation fitness: box fitness + mask fitness",
    "threshold_selection": "Maximum validation mean foreground Dice; tie: higher threshold",
    "sam_model": "sam_b.pt", "sam_examples": 6,
    "augmentation": {"hsv_h": 0.015, "hsv_s": 0.4, "hsv_v": 0.3,
                     "degrees": 10.0, "translate": 0.1, "scale": 0.2,
                     "fliplr": 0.5, "flipud": 0.0, "mosaic": 0.0, "mixup": 0.0},
}
EXPERIMENT = "m5_yolo11n_seg_v1"
RUN_FINAL_TEST = True  # Executes only after validation has fixed the model and threshold.
FORCE_FULL_DECODE = False  # Previously verified files can reuse the full decoding audit.
WANDB_MODE = "online"  # Use "offline" only when authentication/network is unavailable; sync later.
WANDB_ENTITY = "alharby6626-a"
WANDB_PROJECT = "rdd2022-road-damage"
random.seed(PROTOCOL["seed"])
np.random.seed(PROTOCOL["seed"])
torch.manual_seed(PROTOCOL["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(PROTOCOL["seed"])
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (select a GPU before training)")
print("Sampling:", len(PROTOCOL["frame_indices"]), "fixed frames per video in every split")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_csv(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not records:
        raise ValueError(f"Refusing to write an empty table: {path}")
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

def read_csv(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


## 3. Google Drive Connection

We will connect Google Drive and locate the project folder and dataset archive. We will save the model and results inside the M5_Segmentation folder.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

my_drive = Path("/content/drive/MyDrive")
projects = [p for p in my_drive.iterdir() if p.is_dir() and p.name.strip() == "AI Project"]
if len(projects) != 1:
    raise RuntimeError("Expected exactly one AI Project folder/shortcut in MyDrive. Add the shared folder shortcut, then rerun.")
project_dir = projects[0]  # Handles the existing trailing space in this folder's name.
m5_dir = project_dir / "M5_Segmentation"
zip_path = project_dir / "pothole_video.zip"
if not zip_path.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {zip_path}")
for name in ("data", "reports", "models", "runs", "exports"):
    (m5_dir / name).mkdir(parents=True, exist_ok=True)
local_root = Path("/content/m5_local")
local_root.mkdir(parents=True, exist_ok=True)
print("Project:", project_dir)
print("M5:", m5_dir)
print("Archive size (MiB):", round(zip_path.stat().st_size / 1024**2, 1))


## 4. Video and Mask Pairs

We will check that each RGB video has a matching mask video. We will also confirm the number of videos in each split and check whether any video IDs appear in more than one split.

In [ ]:
def inventory_archive(path):
    inventory = defaultdict(dict)
    issues = []
    with zipfile.ZipFile(path) as archive:
        for item in archive.infolist():
            parts = PurePosixPath(item.filename).parts
            if item.is_dir() or any(p.startswith(".") or p == "__MACOSX" for p in parts):
                continue
            if not item.filename.lower().endswith(".mp4"):
                continue
            if (len(parts) != 4 or parts[0] != "pothole_video" or
                parts[1] not in ("train", "val", "test") or parts[2] not in ("rgb", "mask")):
                issues.append(f"Unexpected video path: {item.filename}")
                continue
            split, kind, video_id = parts[1], parts[2], Path(parts[3]).stem
            if not re.fullmatch(r"[A-Za-z0-9_-]+", video_id):
                issues.append(f"Invalid video ID: {video_id}")
                continue
            key = (split, video_id)
            if kind in inventory[key]:
                issues.append(f"Duplicate archive member: {item.filename}")
            inventory[key][kind] = item.filename
    rows = []
    for (split, video_id), files in sorted(inventory.items()):
        if set(files) != {"rgb", "mask"}:
            issues.append(f"Unpaired video: {split}/{video_id}")
        else:
            rows.append({"split": split, "video_id": video_id,
                         "rgb_archive_path": files["rgb"], "mask_archive_path": files["mask"]})
    split_ids = {s: {r["video_id"] for r in rows if r["split"] == s} for s in ("train", "val", "test")}
    for first, second in combinations(split_ids, 2):
        overlap = split_ids[first] & split_ids[second]
        if overlap:
            issues.append(f"Shared IDs between {first}/{second}: {sorted(overlap)}")
    counts = {s: len(ids) for s, ids in split_ids.items()}
    if counts != {"train": 372, "val": 124, "test": 123}:
        issues.append(f"Counts differ from the verified version-3 archive: {counts}")
    return rows, {"matched_pairs": len(rows), "counts": counts, "issues": issues,
                  "status": "PAIRING_PASSED" if not issues else "REVIEW_REQUIRED"}

rows, pairing = inventory_archive(zip_path)
save_json(m5_dir / "reports/archive_pairing_audit.json", pairing)
if pairing["issues"]:
    raise RuntimeError(pairing["issues"])
write_csv(m5_dir / "data/video_manifest.csv", rows)
display(pd.DataFrame([{"split": s, "paired_videos": n} for s, n in pairing["counts"].items()]))
print("Total paired videos:", len(rows))


## 5. File Integrity

We will verify that the video files match the source archive and prepare local copies for faster processing. We will then save the experiment settings together with the dataset fingerprint.

In [ ]:
import csv
import hashlib
import json
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from zipfile import ZipFile


def prepare_video_files(rows, source_zip, cache_root):
    source_zip, cache_root = Path(source_zip), Path(cache_root)
    jobs = []
    for row in rows:
        split, video_id = row["split"], row["video_id"]
        if split not in ("train", "val", "test") or not re.fullmatch(r"[A-Za-z0-9_-]+", video_id):
            raise ValueError(f"Invalid manifest entry: {split}/{video_id}")
        for kind in ("rgb", "mask"):
            member = row[f"{kind}_archive_path"]
            expected = f"pothole_video/{split}/{kind}/{video_id}.mp4"
            if member != expected or len(PurePosixPath(member).parts) != 4:
                raise ValueError(f"Unexpected archive path: {member}")
            jobs.append((split, video_id, kind, member))
    if not jobs or len({job[3] for job in jobs}) != len(jobs):
        raise ValueError("Manifest is empty or contains repeated file paths.")

    cache_root.mkdir(parents=True, exist_ok=True)
    with ZipFile(source_zip) as archive:
        counts = Counter(archive.namelist())
        if any(counts[job[3]] != 1 for job in jobs):
            raise ValueError("A required archive member is missing or duplicated.")
        expanded_bytes = sum(archive.getinfo(job[3]).file_size for job in jobs)
    needed = 2 * source_zip.stat().st_size + expanded_bytes + 64 * 1024**2
    if shutil.disk_usage(cache_root).free < needed:
        raise RuntimeError("Not enough free Colab disk space for preparation.")

    print("Copying the archive to Colab local storage...", flush=True)
    local_zip = cache_root / "pothole_video.zip"
    temporary_zip = cache_root / "pothole_video.zip.part"
    archive_hash = hashlib.sha256()
    with source_zip.open("rb") as source, temporary_zip.open("wb") as target:
        for block in iter(lambda: source.read(4 * 1024**2), b""):
            target.write(block)
            archive_hash.update(block)
    temporary_zip.replace(local_zip)

    fingerprints = []
    with ZipFile(local_zip) as archive:
        for number, (split, video_id, kind, member) in enumerate(jobs, 1):
            destination = cache_root / "videos" / split / kind / f"{video_id}.mp4"
            destination.parent.mkdir(parents=True, exist_ok=True)
            digest = hashlib.sha256()
            size = 0
            with archive.open(member) as source, destination.open("wb") as target:
                for block in iter(lambda: source.read(4 * 1024**2), b""):
                    target.write(block)
                    digest.update(block)
                    size += len(block)
            if size != archive.getinfo(member).file_size:
                raise RuntimeError(f"Incomplete extraction: {member}")
            fingerprints.append({
                "split": split, "video_id": video_id, "kind": kind,
                "archive_member": member, "local_path": str(destination),
                "bytes": size, "sha256": digest.hexdigest(),
            })
            if number % 100 == 0 or number == len(jobs):
                print(f"Verified files: {number}/{len(jobs)}", flush=True)
    return fingerprints, archive_hash.hexdigest()


integrity_path = m5_dir / "reports/video_file_integrity_audit.json"
fingerprint_path = m5_dir / "data/video_file_fingerprints.csv"
previous_integrity = read_json(integrity_path) if integrity_path.exists() else {}
print("Checking the source archive fingerprint...", flush=True)
archive_sha256 = sha256_file(zip_path)
same_archive = previous_integrity.get("archive_sha256") == archive_sha256
fingerprints = read_csv(fingerprint_path) if same_archive and fingerprint_path.exists() else []
expected_paths = {r[f"{kind}_archive_path"] for r in rows for kind in ("rgb", "mask")}
cache_valid = (len(fingerprints) == 2 * len(rows) and
               {r["archive_member"] for r in fingerprints} == expected_paths)
if cache_valid:
    print("Checking cached video files...", flush=True)
    for item in fingerprints:
        path = local_root / "videos" / item["split"] / item["kind"] / f"{item['video_id']}.mp4"
        if not path.is_file() or path.stat().st_size != int(item["bytes"]) or sha256_file(path) != item["sha256"]:
            cache_valid = False
            break
if not cache_valid:
    fingerprints, copied_sha = prepare_video_files(rows, zip_path, local_root)
    if copied_sha != archive_sha256:
        raise RuntimeError("The source archive changed while it was being copied.")
    write_csv(fingerprint_path, fingerprints)
groups = defaultdict(list)
for item in fingerprints:
    if item["kind"] == "rgb":
        groups[item["sha256"]].append({"split": item["split"], "video_id": item["video_id"]})
duplicates = [g for g in groups.values() if len(g) > 1]
cross_split = [g for g in duplicates if len({i["split"] for i in g}) > 1]
save_json(integrity_path, {"archive_sha256": archive_sha256, "verified_files": len(fingerprints),
    "verified_pairs": len(rows), "identical_rgb_file_groups": duplicates,
    "cross_split_identical_rgb_file_groups": cross_split,
    "status": "REVIEW_DUPLICATES" if duplicates else "FILE_CHECKS_PASSED"})
if cross_split:
    raise RuntimeError("Identical RGB files cross dataset splits. Resolve leakage before training.")
print("Verified files:", len(fingerprints), "| Local cache reused:", cache_valid)

# A protocol cannot silently change after a run has started.
run_dir = m5_dir / "runs" / EXPERIMENT
run_dir.mkdir(parents=True, exist_ok=True)
bound_protocol = {**PROTOCOL, "archive_sha256": archive_sha256}
protocol_hash = object_hash(bound_protocol)
protocol_path = run_dir / "protocol.json"
if protocol_path.exists() and read_json(protocol_path) != bound_protocol:
    raise RuntimeError("This experiment already has a different protocol. Do not change it after testing.")
save_json(protocol_path, bound_protocol)
result_dir = run_dir / "evaluation"
result_dir.mkdir(exist_ok=True)
print("Protocol fingerprint:", protocol_hash)


## 6. Video Frame Checks

We will check that the RGB and mask videos can be read and have matching frame counts, dimensions, and frame rates. We will also check for empty or fully filled masks and reuse earlier verified checks when the data is unchanged.

In [ ]:
import csv
import json
from pathlib import Path
import cv2


def inspect_video_pair(rgb_path, mask_path, expected_frames=48):
    rgb_cap = cv2.VideoCapture(str(rgb_path))
    mask_cap = cv2.VideoCapture(str(mask_path))
    issues = set()
    areas = []
    rgb_count = mask_count = empty_masks = full_masks = 0
    gray_min, gray_max = 255, 0
    try:
        def metadata(cap):
            return {
                "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
                "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                "fps": float(cap.get(cv2.CAP_PROP_FPS)),
            }
        rgb_meta, mask_meta = metadata(rgb_cap), metadata(mask_cap)
        if not rgb_cap.isOpened():
            issues.add("RGB video cannot be opened")
        if not mask_cap.isOpened():
            issues.add("Mask video cannot be opened")

        while True:
            rgb_ok, rgb = rgb_cap.read()
            mask_ok, mask = mask_cap.read()
            if not rgb_ok and not mask_ok:
                break
            if rgb_ok:
                rgb_count += 1
                if rgb.shape[:2] != (rgb_meta["height"], rgb_meta["width"]):
                    issues.add("RGB frame dimensions disagree with metadata")
            if mask_ok:
                mask_count += 1
                if mask.shape[:2] != (mask_meta["height"], mask_meta["width"]):
                    issues.add("Mask frame dimensions disagree with metadata")
                gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
                low, high, _, _ = cv2.minMaxLoc(gray)
                gray_min, gray_max = min(gray_min, int(low)), max(gray_max, int(high))
                foreground = cv2.compare(gray, 127, cv2.CMP_GT)
                positive = cv2.countNonZero(foreground)
                areas.append(positive / gray.size)
                empty_masks += int(positive == 0)
                full_masks += int(positive == gray.size)
            if rgb_ok and mask_ok and rgb.shape[:2] != mask.shape[:2]:
                issues.add("RGB and mask frame dimensions differ")

        if rgb_count != rgb_meta["frames"]:
            issues.add("Decoded RGB frame count differs from metadata")
        if mask_count != mask_meta["frames"]:
            issues.add("Decoded mask frame count differs from metadata")
        if rgb_count != mask_count:
            issues.add("Decoded RGB and mask frame counts differ")
        if rgb_count != expected_frames or mask_count != expected_frames:
            issues.add(f"Expected {expected_frames} frames in each video")
        if rgb_meta["fps"] <= 0 or mask_meta["fps"] <= 0:
            issues.add("Invalid frame-rate metadata")
        elif abs(rgb_meta["fps"] - mask_meta["fps"]) >= 0.01:
            issues.add("RGB and mask frame rates differ")

        return {
            "rgb_frames_reported": rgb_meta["frames"],
            "mask_frames_reported": mask_meta["frames"],
            "rgb_frames_decoded": rgb_count,
            "mask_frames_decoded": mask_count,
            "rgb_width": rgb_meta["width"], "rgb_height": rgb_meta["height"],
            "mask_width": mask_meta["width"], "mask_height": mask_meta["height"],
            "rgb_fps": rgb_meta["fps"], "mask_fps": mask_meta["fps"],
            "mask_gray_min": gray_min if areas else None,
            "mask_gray_max": gray_max if areas else None,
            "minimum_foreground_fraction": min(areas) if areas else None,
            "maximum_foreground_fraction": max(areas) if areas else None,
            "mean_foreground_fraction": sum(areas) / len(areas) if areas else None,
            "empty_mask_frames": empty_masks, "full_mask_frames": full_masks,
            "issues": "; ".join(sorted(issues)),
        }
    finally:
        rgb_cap.release()
        mask_cap.release()


decode_path = m5_dir / "reports/video_decode_audit.json"
metadata_path = m5_dir / "data/video_decode_metadata.csv"
old_decode = read_json(decode_path) if decode_path.exists() else {}
metadata = read_csv(metadata_path) if metadata_path.exists() else []
expected_ids = {(r["split"], r["video_id"]) for r in rows}
reuse_decode = (same_archive and not FORCE_FULL_DECODE and
    old_decode.get("status") == "VIDEO_CHECKS_PASSED" and
    len(metadata) == len(rows) and {(r["split"], r["video_id"]) for r in metadata} == expected_ids and
    all(int(r["rgb_frames_decoded"]) == 48 and int(r["mask_frames_decoded"]) == 48
        and not r["issues"] and not int(r["empty_mask_frames"]) and not int(r["full_mask_frames"])
        for r in metadata))
if not reuse_decode:
    metadata = []
    print("Fully decoding every RGB/mask video pair...", flush=True)
    for number, row in enumerate(rows, 1):
        root = local_root / "videos" / row["split"]
        item = {"split": row["split"], "video_id": row["video_id"]}
        item.update(inspect_video_pair(root / "rgb" / f"{row['video_id']}.mp4",
                                      root / "mask" / f"{row['video_id']}.mp4"))
        metadata.append(item)
        if number == 1 or number % 25 == 0 or number == len(rows):
            print(f"Decoded pairs: {number}/{len(rows)}", flush=True)
    write_csv(metadata_path, metadata)
problem_pairs = [r for r in metadata if r["issues"]]
unusual = [r for r in metadata if int(r["empty_mask_frames"]) or int(r["full_mask_frames"])]
decode_report = {"archive_sha256": archive_sha256, "checked_pairs": len(metadata),
    "rgb_frames_decoded": sum(int(r["rgb_frames_decoded"]) for r in metadata),
    "mask_frames_decoded": sum(int(r["mask_frames_decoded"]) for r in metadata),
    "problem_pairs": problem_pairs, "pairs_with_empty_or_full_masks": unusual,
    "status": "REVIEW_REQUIRED" if problem_pairs or unusual else "VIDEO_CHECKS_PASSED",
    "reused_verified_audit": reuse_decode, "reference_mask_rule": "grayscale > 127",
    "visual_alignment": "Technical agreement does not prove every mask boundary is correct."}
save_json(decode_path, decode_report)
print(json.dumps({k: v for k, v in decode_report.items() if not isinstance(v, list)}, indent=2))
if problem_pairs or unusual:
    raise RuntimeError("The full decoding audit needs review before model development.")


## 7. Similarity Between Splits

We will compare representative frames to look for similar videos across the splits. Possible matches will be displayed for review before training; this check may not detect every repeated pothole.

In [ ]:
import csv
import hashlib
import json
from collections import defaultdict
from itertools import combinations
from pathlib import Path
import cv2
import numpy as np


def frame_hashes(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA)
    low = cv2.dct(small.astype(np.float32))[:8, :8].flatten()
    bits = low > np.median(low[1:])
    bits[0] = False
    phash = np.packbits(bits).tobytes().hex()
    small = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA)
    dhash = np.packbits((small[:, 1:] > small[:, :-1]).flatten()).tobytes().hex()
    exact = hashlib.sha256(frame.tobytes()).hexdigest()
    return {"phash": phash, "dhash": dhash, "pixel_sha256": exact}


def compare_signatures(first, second, frame_indices=(0, 24, 47), max_distance=4):
    phash_distances, dhash_distances, matches = [], [], []
    for frame_index in frame_indices:
        a, b = first["frames"][str(frame_index)], second["frames"][str(frame_index)]
        pd = (int(a["phash"], 16) ^ int(b["phash"], 16)).bit_count()
        dd = (int(a["dhash"], 16) ^ int(b["dhash"], 16)).bit_count()
        phash_distances.append(pd)
        dhash_distances.append(dd)
        if pd <= max_distance and dd <= max_distance:
            matches.append(frame_index)
    return matches, phash_distances, dhash_distances


def read_video_frame(split, video_id, kind, index):
    path = local_root / "videos" / split / kind / f"{video_id}.mp4"
    cap = cv2.VideoCapture(str(path))
    try:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ok, frame = cap.read()
        if not ok or int(round(cap.get(cv2.CAP_PROP_POS_FRAMES))) - 1 != int(index):
            raise RuntimeError(f"Cannot read the requested frame: {path}, {index}")
        return frame
    finally:
        cap.release()

screen_path = m5_dir / "reports/visual_duplicate_screening.json"
screen = read_json(screen_path) if screen_path.exists() else {}
if screen.get("archive_sha256") != archive_sha256:
    signatures, hash_records = [], []
    exact_groups = defaultdict(list)
    for number, row in enumerate(rows, 1):
        signature = {"split": row["split"], "video_id": row["video_id"], "frames": {}}
        for frame_index in (0, 24, 47):
            hashes = frame_hashes(read_video_frame(row["split"], row["video_id"], "rgb", frame_index))
            record = {"split": row["split"], "video_id": row["video_id"], "frame_index": frame_index}
            signature["frames"][str(frame_index)] = hashes
            hash_records.append({**record, **hashes})
            exact_groups[hashes["pixel_sha256"]].append(record)
        signatures.append(signature)
        if number % 50 == 0 or number == len(rows):
            print(f"Similarity screening: {number}/{len(rows)}", flush=True)
    candidates = {}
    for group in exact_groups.values():
        for a, b in combinations(group, 2):
            if a["split"] != b["split"]:
                key = "|".join(sorted([f"{a['split']}/{a['video_id']}", f"{b['split']}/{b['video_id']}"]))
                candidates[key] = {"key": key, "first": a, "second": b, "reason": "identical decoded frame"}
    for a, b in combinations(signatures, 2):
        if a["split"] == b["split"]:
            continue
        matches, pdist, ddist = compare_signatures(a, b)
        if len(matches) >= 2:
            key = "|".join(sorted([f"{a['split']}/{a['video_id']}", f"{b['split']}/{b['video_id']}"]))
            candidates.setdefault(key, {"key": key,
                "first": {"split": a["split"], "video_id": a["video_id"], "frame_index": 24},
                "second": {"split": b["split"], "video_id": b["video_id"], "frame_index": 24},
                "reason": "pHash and dHash distance <= 4 on at least two of three aligned frames"})
    screen = {"archive_sha256": archive_sha256, "videos_checked": len(rows),
        "frame_indices": [0, 24, 47], "candidates": list(candidates.values()),
        "limitations": "Heuristic screening can miss different viewpoints, temporal shifts and crops of the same pothole."}
    write_csv(m5_dir / "data/representative_frame_fingerprints.csv", hash_records)
    save_json(screen_path, screen)

print("Cross-split candidate pairs:", len(screen["candidates"]))
# Review candidates; never automatically delete videos or change the published split.
for candidate in screen["candidates"][:12]:
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    for ax, side in zip(axes, ("first", "second")):
        item = candidate[side]
        frame = read_video_frame(item["split"], item["video_id"], "rgb", item["frame_index"])
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{item['split']}/{item['video_id']}")
        ax.axis("off")
    fig.suptitle(candidate["key"])
    plt.show()
    plt.close(fig)
if screen["candidates"]:
    raise RuntimeError("Potential split leakage found. Stop and review visual_duplicate_screening.json and the displayed pairs with your teammate before training. Do not simply bypass this check.")
print("No candidates found by this screen. This is not proof that every pothole identity is independent.")


## 8. Frame and Label Preparation

We will extract 12 fixed frames from each video and resize them to 640 x 640. We will prepare polygon labels for YOLO, compare them with the reference masks, and keep the reference masks for final evaluation.

In [ ]:
def component_masks(mask):
    count, labels = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
    return [(labels == label) for label in range(1, count)]

def mask_to_polygons(mask):
    polygons = []
    for component in component_masks(mask):
        contours, _ = cv2.findContours(component.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            points = contour[:, 0, :].astype(float)
            if len(points) < 3 or cv2.contourArea(contour) == 0:
                # A tiny component still receives a polygon, rather than being silently discarded.
                ys, xs = np.where(component)
                x1, y1 = max(0, xs.min() - 0.5), max(0, ys.min() - 0.5)
                x2, y2 = min(mask.shape[1] - 0.01, xs.max() + 0.5), min(mask.shape[0] - 0.01, ys.max() + 0.5)
                points = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
            polygons.append(points)
    return polygons

def polygon_union(polygons, shape):
    canvas = np.zeros(shape, np.uint8)
    for points in polygons:
        cv2.fillPoly(canvas, [np.rint(points).astype(np.int32)], 1)
    return canvas.astype(bool)

def binary_scores(reference, prediction):
    reference, prediction = np.asarray(reference, bool), np.asarray(prediction, bool)
    if reference.shape != prediction.shape:
        raise ValueError("Reference and prediction shapes differ.")
    tp = int(np.count_nonzero(reference & prediction))
    fp = int(np.count_nonzero(~reference & prediction))
    fn = int(np.count_nonzero(reference & ~prediction))
    tn = reference.size - tp - fp - fn
    safe = lambda n, d: n / d if d else 1.0
    fg_iou, bg_iou = safe(tp, tp + fp + fn), safe(tn, tn + fp + fn)
    true_area = 100.0 * (tp + fn) / reference.size
    predicted_area = 100.0 * (tp + fp) / reference.size
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "foreground_iou": fg_iou, "background_iou": bg_iou,
        "mean_iou": (fg_iou + bg_iou) / 2, "dice": safe(2 * tp, 2 * tp + fp + fn),
        "true_area_pct": true_area, "predicted_area_pct": predicted_area,
        "area_error_pp": predicted_area - true_area,
        "absolute_area_error_pp": abs(predicted_area - true_area)}

dataset_dir = local_root / f"frames_{protocol_hash[:12]}"
frame_manifest_path = m5_dir / "data" / f"frame_manifest_{protocol_hash[:12]}.csv"
complete_path = dataset_dir / "complete.json"
frame_rows = read_csv(frame_manifest_path) if frame_manifest_path.exists() else []
reuse_frames = (complete_path.exists() and read_json(complete_path).get("protocol_hash") == protocol_hash
    and len(frame_rows) == len(rows) * len(PROTOCOL["frame_indices"]))
if reuse_frames:
    for row in frame_rows:
        for field in ("image_path", "mask_path", "label_path"):
            path = dataset_dir / row[field]
            if not path.is_file() or sha256_file(path) != row[field.replace("_path", "_sha256")]:
                reuse_frames = False
                break
        if not reuse_frames:
            break
if not reuse_frames:
    if shutil.disk_usage(local_root).free < 12 * 1024**3:
        raise RuntimeError("At least 12 GiB of local free space is needed for frame preparation.")
    frame_rows = []
    for number, video in enumerate(rows, 1):
        split, video_id = video["split"], video["video_id"]
        for folder in ("images", "masks", "labels"):
            (dataset_dir / folder / split).mkdir(parents=True, exist_ok=True)
        rgb_cap = cv2.VideoCapture(str(local_root / "videos" / split / "rgb" / f"{video_id}.mp4"))
        mask_cap = cv2.VideoCapture(str(local_root / "videos" / split / "mask" / f"{video_id}.mp4"))
        try:
            for index in range(48):
                ok_rgb, rgb = rgb_cap.read()
                ok_mask, raw_mask = mask_cap.read()
                if not ok_rgb or not ok_mask:
                    raise RuntimeError(f"Unexpected video decoding failure: {split}/{video_id}/{index}")
                if index not in PROTOCOL["frame_indices"]:
                    continue
                original_h, original_w = rgb.shape[:2]
                rgb = cv2.resize(rgb, (PROTOCOL["imgsz"], PROTOCOL["imgsz"]), interpolation=cv2.INTER_AREA)
                reference = cv2.cvtColor(raw_mask, cv2.COLOR_BGR2GRAY) > PROTOCOL["mask_threshold"]
                reference = cv2.resize(reference.astype(np.uint8), (PROTOCOL["imgsz"], PROTOCOL["imgsz"]),
                                       interpolation=cv2.INTER_NEAREST).astype(bool)
                if not reference.any() or reference.all():
                    raise RuntimeError(f"Empty/full reference after resizing: {split}/{video_id}/{index}")
                name = f"{video_id}_f{index:02d}"
                image_rel, mask_rel = f"images/{split}/{name}.png", f"masks/{split}/{name}.png"
                label_rel = f"labels/{split}/{name}.txt"
                if not cv2.imwrite(str(dataset_dir / image_rel), rgb):
                    raise IOError("Failed to write image")
                if not cv2.imwrite(str(dataset_dir / mask_rel), reference.astype(np.uint8) * 255):
                    raise IOError("Failed to write mask")
                polygons = mask_to_polygons(reference)
                lines = []
                for polygon in polygons:
                    normalized = polygon / np.array([reference.shape[1], reference.shape[0]])
                    lines.append("0 " + " ".join(f"{v:.7f}" for v in normalized.flatten()))
                (dataset_dir / label_rel).write_text("\n".join(lines) + "\n", encoding="utf-8")
                approximation = binary_scores(reference, polygon_union(polygons, reference.shape))
                frame_rows.append({"split": split, "video_id": video_id, "frame_index": index,
                    "image_path": image_rel, "mask_path": mask_rel, "label_path": label_rel,
                    "image_sha256": sha256_file(dataset_dir / image_rel),
                    "mask_sha256": sha256_file(dataset_dir / mask_rel),
                    "label_sha256": sha256_file(dataset_dir / label_rel),
                    "original_width": original_w, "original_height": original_h,
                    "evaluation_width": reference.shape[1], "evaluation_height": reference.shape[0],
                    "derived_components": len(component_masks(reference)), "training_polygons": len(polygons),
                    "polygon_conversion_iou": approximation["foreground_iou"]})
        finally:
            rgb_cap.release()
            mask_cap.release()
        if number % 25 == 0 or number == len(rows):
            print(f"Prepared videos: {number}/{len(rows)}", flush=True)
    write_csv(frame_manifest_path, frame_rows)
    save_json(complete_path, {"protocol_hash": protocol_hash, "frames": len(frame_rows)})
frame_df = pd.DataFrame(frame_rows)
frame_df["frame_index"] = frame_df["frame_index"].astype(int)
display(frame_df.groupby("split").agg(videos=("video_id", "nunique"), frames=("video_id", "size")))
conversion = pd.to_numeric(frame_df["polygon_conversion_iou"])
conversion_report = {"frames": len(frame_rows), "minimum_polygon_iou": float(conversion.min()),
    "mean_polygon_iou": float(conversion.mean()), "frames_below_0_95": int((conversion < 0.95).sum()),
    "mean_derived_components": float(pd.to_numeric(frame_df["derived_components"]).mean()),
    "maximum_derived_components": int(pd.to_numeric(frame_df["derived_components"]).max()),
    "frames_with_multiple_components": int((pd.to_numeric(frame_df["derived_components"]) > 1).sum()),
    "reference_preserved": "Final metrics use resized binary reference PNG masks, not polygon labels.",
    "semantic_instance_limitation": "Connected components are derived region IDs; the source has no native instance IDs."}
save_json(run_dir / "polygon_conversion_audit.json", conversion_report)
print(json.dumps(conversion_report, indent=2))
preview_rows = frame_df[frame_df["split"] == "train"].drop_duplicates("video_id").head(3)
fig, axes = plt.subplots(len(preview_rows), 3, figsize=(10, 3.5 * len(preview_rows)), squeeze=False)
for index, (_, record) in enumerate(preview_rows.iterrows()):
    rgb = cv2.cvtColor(cv2.imread(str(dataset_dir / record["image_path"])), cv2.COLOR_BGR2RGB)
    reference = cv2.imread(str(dataset_dir / record["mask_path"]), cv2.IMREAD_GRAYSCALE) > 127
    approximation = polygon_union(mask_to_polygons(reference), reference.shape)
    for ax, image, title in zip(axes[index], [rgb, reference, approximation],
        [f"Training {record['video_id']}", "Reference raster", "Polygon raster approximation"]):
        ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
fig.tight_layout()
fig.savefig(run_dir / "polygon_label_preview.png", dpi=130)
plt.show()
plt.close(fig)
data_config = {"path": str(dataset_dir), "train": "images/train", "val": "images/val",
               "test": "images/test", "names": {0: "pothole"}}
data_yaml = dataset_dir / "data.yaml"
data_yaml.write_text(yaml.safe_dump(data_config), encoding="utf-8")
shutil.copy2(data_yaml, run_dir / "data.yaml")


## 9. SAM Mask Proposals

We will use SAM to generate mask proposals for six training examples, using boxes obtained from their reference masks. We will display each image beside its reference mask and SAM proposal to compare their boundaries.

In [ ]:
sam_dir = run_dir / "sam_review"
sam_dir.mkdir(exist_ok=True)
sam_summary_path = sam_dir / "proposals.json"
training_ids = sorted(r["video_id"] for r in rows if r["split"] == "train")
sam_ids = random.Random(PROTOCOL["seed"]).sample(training_ids, PROTOCOL["sam_examples"])
sam_records = read_json(sam_summary_path) if sam_summary_path.exists() else []
sam_cached = len(sam_records) == len(sam_ids) and all((sam_dir / r["figure"]).exists() and
    (sam_dir / r["proposal_mask"]).exists() for r in sam_records)
if not sam_cached:
    if DEVICE == "cpu":
        raise RuntimeError("Choose a Colab GPU, reconnect and run the notebook again before SAM/training.")
    sam = SAM(PROTOCOL["sam_model"])
    sam_records = []
    for video_id in sam_ids:
        rgb = read_video_frame("train", video_id, "rgb", 24)
        raw = read_video_frame("train", video_id, "mask", 24)
        rgb = cv2.resize(rgb, (PROTOCOL["imgsz"], PROTOCOL["imgsz"]), interpolation=cv2.INTER_AREA)
        reference = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY) > PROTOCOL["mask_threshold"]
        reference = cv2.resize(reference.astype(np.uint8), rgb.shape[1::-1], interpolation=cv2.INTER_NEAREST).astype(bool)
        ys, xs = np.where(reference)
        bbox = [int(xs.min()), int(ys.min()), int(xs.max() + 1), int(ys.max() + 1)]
        result = sam.predict(source=rgb, bboxes=[bbox], device=DEVICE, verbose=False)[0]
        proposal = np.zeros(reference.shape, bool)
        if result.masks is not None:
            for mask in result.masks.data.cpu().numpy():
                resized = cv2.resize(mask.astype(np.uint8), rgb.shape[1::-1], interpolation=cv2.INTER_NEAREST)
                proposal |= resized.astype(bool)
        stats = binary_scores(reference, proposal)
        rgb_display = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        overlay = rgb_display.copy()
        overlay[proposal] = (0.55 * overlay[proposal] + 0.45 * np.array([0, 255, 0])).astype(np.uint8)
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        for ax, image, title in zip(axes, [rgb_display, reference, proposal, overlay],
            [f"Train {video_id}, frame 24", "Original reference", "SAM box-prompt proposal", "Proposal overlay"]):
            ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
            ax.set_title(title)
            ax.axis("off")
        fig.suptitle(f"Reference/SAM IoU: {stats['foreground_iou']:.3f} | Dice: {stats['dice']:.3f}")
        fig.tight_layout()
        figure_name, mask_name = f"sam_{video_id}.png", f"sam_{video_id}_mask.png"
        fig.savefig(sam_dir / figure_name, dpi=120, bbox_inches="tight")
        plt.close(fig)
        cv2.imwrite(str(sam_dir / mask_name), proposal.astype(np.uint8) * 255)
        sam_records.append({"video_id": video_id, "frame_index": 24, "split": "train",
            "figure": figure_name, "proposal_mask": mask_name, "prompt_bbox_from_training_gt": bbox,
            "foreground_iou": stats["foreground_iou"], "dice": stats["dice"],
            "used_to_replace_reference": False})
    save_json(sam_summary_path, sam_records)
    del sam
    gc.collect()
    torch.cuda.empty_cache()
print("SAM proposals saved. Original reference masks remain the training/evaluation targets.")
for record in sam_records:
    display(Image.open(sam_dir / record["figure"]))


## 10. Visual Review

We will review the six SAM proposals and record whether each is acceptable, needs refinement, or should be rejected. We will add a short observation for each example while retaining the original reference masks as training targets.

In [ ]:
import ipywidgets as widgets
from google.colab import output
output.enable_custom_widget_manager()

review_path = sam_dir / "visual_review.json"
previous_review = read_json(review_path) if review_path.exists() else {}
previous_by_id = {r["video_id"]: r for r in previous_review.get("reviews", [])}
reviewer_widget = widgets.Text(value=previous_review.get("reviewer", "Abdulrahman Alsalmi"), description="Reviewer:")
review_controls = []
display(reviewer_widget)
for record in sam_records:
    old = previous_by_id.get(record["video_id"], {})
    verdict = widgets.Dropdown(options=[("Select after visual inspection", "pending"),
        ("Acceptable proposal", "acceptable"), ("Needs refinement", "needs_refinement"),
        ("Reject proposal", "reject")], value=old.get("decision", "pending"), description=record["video_id"])
    note = widgets.Textarea(value=old.get("note", ""), placeholder="Describe boundary alignment, missed area or extra background.",
                            layout=widgets.Layout(width="85%", height="60px"))
    review_controls.append((record, verdict, note))
    display(widgets.VBox([verdict, note]))
save_button = widgets.Button(description="Save visual review", button_style="success")
review_output = widgets.Output()

def save_visual_review(_):
    with review_output:
        review_output.clear_output()
        reviews = [{"video_id": r["video_id"], "frame_index": 24, "decision": decision.value,
                    "note": note.value.strip(), "reference_retained": True}
                   for r, decision, note in review_controls]
        complete = bool(reviewer_widget.value.strip()) and all(r["decision"] != "pending" and r["note"] for r in reviews)
        save_json(review_path, {"reviewer": reviewer_widget.value.strip(), "reviews": reviews,
            "complete": complete, "saved_utc": datetime.now(timezone.utc).isoformat(),
            "scope": "Six training-frame proposals only; not an exhaustive label audit."})
        print("Review saved:", "COMPLETE" if complete else "PENDING decisions/notes")
        print("After saving, rerun the report and package cells at the end; training/test do not need to run again.")

save_button.on_click(save_visual_review)
display(save_button, review_output)
print("Inspect the six figures above, select a decision and write one observation for each, then save.")
print("Training can proceed with the unchanged reference masks while this review is pending.")


## 11. Model Training

We will fine-tune YOLO11n-seg on the training images for 30 epochs and measure its validation performance after each epoch. We will save checkpoints in Drive and keep the model with the best combined box and mask validation score.

In [ ]:
training_dir = run_dir / "training"
training_complete = run_dir / "training_complete.json"
selection_path = result_dir / "selection.json"
test_lock_path = result_dir / "test_lock.json"
best_path = m5_dir / "models" / f"{EXPERIMENT}_best.pt"

if training_complete.exists():
    trained = read_json(training_complete)
    if not best_path.exists() or sha256_file(best_path) != trained["checkpoint_sha256"]:
        raise RuntimeError("The saved selected checkpoint is missing or changed. Restore the original checkpoint.")
    print("Reusing the completed training run and its saved checkpoint.")
else:
    if test_lock_path.exists() or selection_path.exists():
        raise RuntimeError("Validation/test has already been locked. Retraining this experiment is not allowed.")
    if DEVICE == "cpu":
        raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun. Full training is not intended for CPU.")
    last_path = training_dir / "weights/last.pt"
    history_path = training_dir / "results.csv"
    already_finished = False
    if last_path.exists():
        model = YOLO(str(last_path))
        if int(model.ckpt.get("epoch", -1)) < 0:
            already_finished = history_path.exists() and len(pd.read_csv(history_path)) >= PROTOCOL["epochs"]
            if not already_finished:
                raise RuntimeError("Checkpoint has no resume state, but completion is unverified. Review results.csv before changing anything.")
        if not already_finished:
            print("Resuming the interrupted run from its last saved epoch.")
            model.train(resume=True, device=DEVICE)
    else:
        model = YOLO(PROTOCOL["model"])
        model.train(data=str(data_yaml), epochs=PROTOCOL["epochs"], imgsz=PROTOCOL["imgsz"],
            batch=PROTOCOL["batch"], workers=PROTOCOL["workers"], device=DEVICE,
            seed=PROTOCOL["seed"], deterministic=True, pretrained=True, freeze=None,
            optimizer=PROTOCOL["optimizer"], lr0=PROTOCOL["lr0"], lrf=PROTOCOL["lrf"], weight_decay=PROTOCOL["weight_decay"],
            patience=PROTOCOL["epochs"], amp=True, cache=False, save=True, save_period=1,
            project=str(run_dir), name="training", exist_ok=True, plots=True, val=True,
            **PROTOCOL["augmentation"])
    source_best = training_dir / "weights/best.pt"
    if not source_best.is_file():
        raise RuntimeError("Training did not produce best.pt.")
    shutil.copy2(source_best, best_path)
    trained = {"checkpoint_sha256": sha256_file(best_path), "protocol_hash": protocol_hash,
        "selection": PROTOCOL["checkpoint_selection"], "epochs_logged": len(pd.read_csv(history_path)),
        "finished_utc": datetime.now(timezone.utc).isoformat()}
    save_json(training_complete, trained)
    del model
    gc.collect()
    torch.cuda.empty_cache()

checkpoint_sha256 = sha256_file(best_path)
history = pd.read_csv(training_dir / "results.csv")
history.columns = history.columns.str.strip()
display(history.tail(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for column in ("train/seg_loss", "val/seg_loss"):
    if column in history:
        axes[0].plot(history["epoch"], history[column], label=column)
for column in ("metrics/mAP50-95(M)", "metrics/mAP50-95(B)"):
    if column in history:
        axes[1].plot(history["epoch"], history[column], label=column)
for ax, title in zip(axes, ["Segmentation loss", "Internal validation AP (polygon targets)"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(run_dir / "learning_curves.png", dpi=150)
plt.show()
plt.close(fig)
print("Saved selected checkpoint:", best_path)


## 12. Evaluation Metrics

We will calculate mask mAP50-95, mean IoU, foreground IoU, Dice, and area estimation error. Mean IoU will include foreground and background, while Dice will measure foreground overlap. Since the dataset provides binary masks without instance IDs, we will use connected foreground regions as derived instances for mask AP.

In [ ]:
from contextlib import redirect_stdout
from pycocotools import mask as coco_mask
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def encode_mask(mask):
    encoded = coco_mask.encode(np.asfortranarray(mask.astype(np.uint8)))
    return {"size": [int(v) for v in encoded["size"]], "counts": encoded["counts"].decode("ascii")}

def decode_mask(encoded):
    return coco_mask.decode({"size": encoded["size"], "counts": encoded["counts"].encode("ascii")}).astype(bool)

def coco_average_precision(ground_truth, predictions):
    # Empty predictions need an explicit empty COCO dataset; loadRes([]) is not supported.
    capture = io.StringIO()
    with redirect_stdout(capture):
        gt = COCO()
        gt.dataset = ground_truth
        gt.createIndex()
        if predictions:
            detections = gt.loadRes(predictions)
        else:
            detections = COCO()
            detections.dataset = {"images": ground_truth["images"],
                "categories": ground_truth["categories"], "annotations": []}
            detections.createIndex()
        evaluator = COCOeval(gt, detections, "segm")
        evaluator.params.imgIds = [r["id"] for r in ground_truth["images"]]
        evaluator.params.catIds = [1]
        evaluator.params.iouThrs = np.linspace(0.5, 0.95, 10)
        evaluator.params.maxDets = [1, 10, 100]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    if evaluator.stats[0] < 0:
        raise RuntimeError("COCO AP is undefined: check images and ground-truth instances.")
    return {"mask_map50_95": float(evaluator.stats[0]), "mask_ap50": float(evaluator.stats[1]),
            "mask_ap75": float(evaluator.stats[2])}, capture.getvalue()

def extract_prediction_instances(result, shape):
    instances = []
    if result.masks is None:
        return instances
    masks = result.masks.data.detach().cpu().numpy()
    scores = result.boxes.conf.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    if len(masks) != len(scores):
        raise RuntimeError("Prediction mask/score counts do not match.")
    for mask, score, class_id in zip(masks, scores, classes):
        if class_id != 0:
            raise RuntimeError("The trained M5 model must have only class 0: pothole.")
        if mask.shape != shape:
            raise RuntimeError("Prediction mask dimensions differ from reference. retina_masks=True is required.")
        mask = mask > 0.5
        if mask.any():
            instances.append({"score": float(score), "segmentation": encode_mask(mask)})
    return instances

def union_instances(instances, shape, threshold):
    union = np.zeros(shape, bool)
    for instance in instances:
        if instance["score"] >= threshold:
            union |= decode_mask(instance["segmentation"])
    return union

def aggregate_scores(records):
    if not records:
        raise ValueError("No images were evaluated.")
    frame = pd.DataFrame(records)
    return {"frames": len(frame), "videos": int(frame["video_id"].nunique()),
        "mean_iou": float(frame["mean_iou"].mean()),
        "foreground_iou": float(frame["foreground_iou"].mean()),
        "dice": float(frame["dice"].mean()),
        "severity_mae_pp": float(frame["absolute_area_error_pp"].mean()),
        "severity_rmse_pp": float(np.sqrt(np.mean(frame["area_error_pp"] ** 2))),
        "severity_bias_pp": float(frame["area_error_pp"].mean())}

def evaluation_fingerprint(split, thresholds):
    return object_hash({"protocol_hash": protocol_hash, "checkpoint": checkpoint_sha256,
                        "split": split, "thresholds": thresholds, "metric_version": "raster-cc8-v1"})

def evaluate_split(split, thresholds):
    if split not in ("val", "test"):
        raise ValueError("Only val/test are evaluation splits.")
    thresholds = [float(t) for t in thresholds]
    fingerprint = evaluation_fingerprint(split, thresholds)
    split_dir = result_dir / split
    split_dir.mkdir(exist_ok=True)
    cache_dir = split_dir / "frame_predictions"
    cache_dir.mkdir(exist_ok=True)
    summary_file = split_dir / "summary.json"
    if summary_file.exists():
        summary = read_json(summary_file)
        if summary["fingerprint"] != fingerprint:
            raise RuntimeError(f"An existing {split} evaluation uses different settings.")
        print(f"Reusing completed {split} evaluation.")
        return summary
    subset = [r for r in frame_rows if r["split"] == split]
    ground_truth = {"info": {"description": "Derived 8-connected regions of binary reference masks at 640x640"},
        "images": [], "annotations": [], "categories": [{"id": 1, "name": "pothole"}]}
    predictions = []
    threshold_records = {str(t): [] for t in thresholds}
    predictor = None
    annotation_id = 1
    for image_id, row in enumerate(subset, 1):
        image_path = dataset_dir / row["image_path"]
        reference = cv2.imread(str(dataset_dir / row["mask_path"]), cv2.IMREAD_GRAYSCALE)
        if reference is None:
            raise FileNotFoundError(row["mask_path"])
        reference = reference > 127
        h, w = reference.shape
        ground_truth["images"].append({"id": image_id, "file_name": row["image_path"], "height": h, "width": w})
        for component in component_masks(reference):
            encoded = encode_mask(component)
            ground_truth["annotations"].append({"id": annotation_id, "image_id": image_id, "category_id": 1,
                "segmentation": encoded, "area": int(component.sum()),
                "bbox": [float(v) for v in coco_mask.toBbox(encoded)], "iscrowd": 0})
            annotation_id += 1
        cache_path = cache_dir / f"{row['video_id']}_f{int(row['frame_index']):02d}.json"
        frame_key = object_hash({"fingerprint": fingerprint, "image": row["image_sha256"], "mask": row["mask_sha256"]})
        cached = read_json(cache_path) if cache_path.exists() else {}
        if cached.get("frame_key") != frame_key:
            if cached:
                raise RuntimeError("A cached frame belongs to a different evaluation. Do not overwrite final test evidence.")
            if predictor is None:
                predictor = YOLO(str(best_path))
            result = predictor.predict(source=str(image_path), imgsz=PROTOCOL["imgsz"],
                conf=PROTOCOL["eval_conf_floor"], iou=PROTOCOL["nms_iou"], max_det=PROTOCOL["max_det"],
                retina_masks=True, device=DEVICE, verbose=False, augment=False)[0]
            cached = {"frame_key": frame_key, "instances": extract_prediction_instances(result, reference.shape)}
            save_json(cache_path, cached)
        instances = cached["instances"]
        predictions.extend({"image_id": image_id, "category_id": 1, **instance} for instance in instances)
        for threshold in thresholds:
            union = union_instances(instances, reference.shape, threshold)
            record = {"video_id": row["video_id"], "frame_index": int(row["frame_index"]),
                "image_id": image_id, "image_path": row["image_path"], "mask_path": row["mask_path"],
                "threshold": threshold, **binary_scores(reference, union)}
            threshold_records[str(threshold)].append(record)
        if image_id == 1 or image_id % 100 == 0 or image_id == len(subset):
            print(f"{split}: {image_id}/{len(subset)} frames", flush=True)
    save_json(split_dir / "coco_reference.json", ground_truth)
    save_json(split_dir / "coco_predictions.json", predictions)
    ap, ap_log = coco_average_precision(ground_truth, predictions)
    (split_dir / "coco_evaluation.txt").write_text(ap_log, encoding="utf-8")
    threshold_summary = []
    for threshold in thresholds:
        records = threshold_records[str(threshold)]
        write_csv(split_dir / f"per_frame_t{threshold:.2f}.csv", records)
        threshold_summary.append({"threshold": threshold, **aggregate_scores(records)})
    summary = {"split": split, "fingerprint": fingerprint, "protocol_hash": protocol_hash,
        "checkpoint_sha256": checkpoint_sha256, "ap": ap, "thresholds": threshold_summary,
        "derived_gt_instances": len(ground_truth["annotations"]),
        "ap_scope": "COCO segmentation AP of derived connected regions; not a native-instance benchmark",
        "metric_resolution": [PROTOCOL["imgsz"], PROTOCOL["imgsz"]]}
    save_json(summary_file, summary)
    if predictor is not None:
        del predictor
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return summary

print("Evaluation functions ready. No test inference has run in this cell.")


## 13. Validation Results

We will evaluate the selected model on the validation split and compare several confidence thresholds. We will select the threshold with the highest mean foreground Dice and save it for the final test.

In [ ]:
validation_summary = evaluate_split("val", PROTOCOL["threshold_grid"])
table = pd.DataFrame(validation_summary["thresholds"])
display(table[["threshold", "dice", "mean_iou", "foreground_iou", "severity_mae_pp"]])
best_threshold_row = max(validation_summary["thresholds"], key=lambda r: (r["dice"], r["threshold"]))
chosen_threshold = float(best_threshold_row["threshold"])
selection = {"protocol_hash": protocol_hash, "checkpoint_sha256": checkpoint_sha256,
    "threshold": chosen_threshold, "selection_split": "val", "selection_metric": "mean foreground Dice",
    "tie_break": "higher threshold", "validation_fingerprint": validation_summary["fingerprint"],
    "validation_ap": validation_summary["ap"], "validation_operating_metrics": best_threshold_row}
if selection_path.exists() and read_json(selection_path) != selection:
    raise RuntimeError("The stored validation selection differs; do not change a locked experiment.")
save_json(selection_path, selection)
print("Selected operating confidence threshold:", chosen_threshold)
print("Validation reference-mask AP:", validation_summary["ap"])


## 14. Final Test Results

We will evaluate the fixed model and threshold on the 1,476 sampled test frames. We will report the results without using them to change the model or its settings.

In [ ]:
test_summary = None
if RUN_FINAL_TEST:
    if not selection_path.exists() or sha256_file(best_path) != selection["checkpoint_sha256"]:
        raise RuntimeError("A fixed validation selection and unchanged checkpoint are required.")
    lock = {"protocol_hash": protocol_hash, "checkpoint_sha256": checkpoint_sha256,
            "selected_threshold": chosen_threshold, "selection_fingerprint": object_hash(selection)}
    if test_lock_path.exists() and read_json(test_lock_path) != lock:
        raise RuntimeError("The test protocol is already locked to different settings.")
    # Write the lock before any test predictions; interrupted evaluation resumes the same frame cache.
    save_json(test_lock_path, lock)
    test_summary = evaluate_split("test", [chosen_threshold])
    final_results = {**test_summary["ap"], **test_summary["thresholds"][0]}
    display(pd.DataFrame([final_results]))
    print("Final test completed. Do not tune the model or threshold using these results.")
else:
    print("Final test is disabled. Submission remains incomplete until this cell is run with the frozen selection.")


## 15. Prediction and Error Analysis

We will compare successful and difficult predictions with their reference masks. We will highlight missed and extra predicted pixels, analyze a failure case, and measure the error in estimated pothole area.

In [ ]:
figure_dir = run_dir / "figures"
figure_dir.mkdir(exist_ok=True)

def prediction_for_record(record, split, threshold):
    path = result_dir / split / "frame_predictions" / f"{record['video_id']}_f{int(record['frame_index']):02d}.json"
    return union_instances(read_json(path)["instances"], (PROTOCOL["imgsz"], PROTOCOL["imgsz"]), threshold)

failure_analyses = {}
for summary in [validation_summary] + ([test_summary] if test_summary is not None else []):
    split = summary["split"]
    per_frame_path = result_dir / split / f"per_frame_t{chosen_threshold:.2f}.csv"
    measurements = pd.read_csv(per_frame_path, dtype={"video_id": str})
    # Show different videos, so adjacent near-identical frames do not fill the gallery.
    worst = measurements.sort_values("foreground_iou").drop_duplicates("video_id").head(4)
    best = measurements.sort_values("foreground_iou", ascending=False).drop_duplicates("video_id").head(2)
    gallery = pd.concat([worst, best], ignore_index=True)
    fig, axes = plt.subplots(len(gallery), 4, figsize=(13, 3 * len(gallery)))
    for row_index, (_, record) in enumerate(gallery.iterrows()):
        rgb = cv2.cvtColor(cv2.imread(str(dataset_dir / record["image_path"])), cv2.COLOR_BGR2RGB)
        reference = cv2.imread(str(dataset_dir / record["mask_path"]), cv2.IMREAD_GRAYSCALE) > 127
        predicted = prediction_for_record(record, split, chosen_threshold)
        error_overlay = (rgb * 0.4).astype(np.uint8)
        error_overlay[reference & predicted] = [40, 200, 70]  # True positive: green.
        error_overlay[~reference & predicted] = [240, 60, 60]  # False positive: red.
        error_overlay[reference & ~predicted] = [60, 120, 255]  # False negative: blue.
        titles = [f"{split}/{record['video_id']} f{int(record['frame_index'])}", "Reference",
                  f"Prediction | IoU {record['foreground_iou']:.3f}", "TP green / FP red / FN blue"]
        for ax, img, title in zip(axes[row_index], [rgb, reference, predicted, error_overlay], titles):
            ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
            ax.set_title(title, fontsize=10)
            ax.axis("off")
    fig.suptitle("Four lowest-IoU videos, then two highest-IoU videos", fontsize=13)
    fig.tight_layout()
    fig.savefig(figure_dir / f"{split}_qualitative_gallery.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    failed = worst.iloc[0]
    fp, fn, tp = int(failed["fp"]), int(failed["fn"]), int(failed["tp"])
    if fp + fn == 0:
        analysis = "No pixel-level failure was found among these evaluated frames; a failure case must not be invented."
    else:
        behavior = "missed foreground dominates" if fn > fp else "extra predicted foreground dominates"
        missed = 100 * fn / (tp + fn) if tp + fn else 0
        analysis = (f"Video {failed['video_id']}, frame {int(failed['frame_index'])}: foreground IoU "
            f"{failed['foreground_iou']:.4f}, Dice {failed['dice']:.4f}. "
            f"There are {fn:,} missed foreground pixels ({missed:.2f}% of the reference area) and "
            f"{fp:,} extra foreground pixels; {behavior}. "
            f"True area is {failed['true_area_pct']:.2f}% of the image and predicted area is "
            f"{failed['predicted_area_pct']:.2f}%, a signed error of {failed['area_error_pp']:.2f} percentage points. "
            "The error map locates the mismatch. Possible explanations to check visually include ambiguous boundaries, "
            "water/texture variation and detail lost during resizing; these are hypotheses, not established causes. "
            "A future training-only experiment could add reviewed examples with similar appearance or increase input resolution. "
            "The current model and threshold remain fixed; test failures are reporting evidence only.")
    failure_analyses[split] = analysis
    display(Markdown("**Failure analysis:** " + analysis))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(measurements["true_area_pct"], measurements["predicted_area_pct"], alpha=0.25, s=10)
    limit = max(measurements["true_area_pct"].max(), measurements["predicted_area_pct"].max(), 1)
    axes[0].plot([0, limit], [0, limit], "k--")
    axes[0].set(xlabel="Reference area (% of image)", ylabel="Predicted area (% of image)", title=f"{split}: severity proxy")
    axes[1].hist(measurements["foreground_iou"], bins=20, range=(0, 1))
    axes[1].set(xlabel="Foreground IoU", ylabel="Frames", title=f"{split}: overlap distribution")
    fig.tight_layout()
    fig.savefig(figure_dir / f"{split}_area_and_iou.png", dpi=150)
    plt.show()
    plt.close(fig)
    video_table = measurements.groupby("video_id", as_index=False).agg(
        mean_iou=("mean_iou", "mean"), foreground_iou=("foreground_iou", "mean"), dice=("dice", "mean"),
        severity_mae_pp=("absolute_area_error_pp", "mean"), severity_bias_pp=("area_error_pp", "mean"))
    video_table.to_csv(result_dir / split / "per_video_metrics.csv", index=False)
save_json(result_dir / "failure_analyses.json", failure_analyses)


## 16. Model Export

We will save the trained model, selected threshold, environment details, and a helper for making predictions on new images. We will also document how the M5 pothole class corresponds to D40 in M4.

In [ ]:
INFERENCE_SOURCE = '"""Inference-only M5 handoff. No reference masks or M4 model are required."""\nimport json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom ultralytics import YOLO\n\n\nclass PotholeSegmenter:\n    def __init__(self, checkpoint, config, device="cpu"):\n        self.model = YOLO(str(checkpoint))\n        self.config = json.loads(Path(config).read_text(encoding="utf-8"))\n        self.device = device\n\n    def predict(self, image):\n        """Accept an image path or a BGR uint8 array; return masks at the original image size."""\n        bgr = cv2.imread(str(image)) if isinstance(image, (str, Path)) else np.asarray(image)\n        if bgr is None or bgr.ndim != 3 or bgr.shape[2] != 3:\n            raise ValueError("Expected a readable three-channel BGR image.")\n        original_h, original_w = bgr.shape[:2]\n        size = int(self.config["imgsz"])\n        resized = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)\n        result = self.model.predict(source=resized, imgsz=size,\n            conf=self.config["threshold"], iou=self.config["nms_iou"],\n            max_det=self.config["max_det"], retina_masks=True, device=self.device,\n            verbose=False, augment=False)[0]\n        union = np.zeros((original_h, original_w), bool)\n        instances = []\n        if result.masks is not None:\n            for mask, score, category in zip(result.masks.data.cpu().numpy(),\n                    result.boxes.conf.cpu().numpy(), result.boxes.cls.cpu().numpy()):\n                if int(category) != 0:\n                    raise ValueError("The M5 checkpoint must use class 0 for pothole.")\n                binary = cv2.resize((mask > 0.5).astype(np.uint8), (original_w, original_h),\n                                    interpolation=cv2.INTER_NEAREST).astype(bool)\n                union |= binary\n                instances.append({"mask": binary, "confidence": float(score),\n                    "m5_class_id": 0, "m4_class_id": 3, "damage_code": "D40"})\n        return {"mask": union, "area_pixels": int(union.sum()),\n            "area_fraction": float(union.mean()), "area_percent": 100 * float(union.mean()),\n            "original_size_hw": [original_h, original_w], "instances": instances}\n'

export_dir = m5_dir / "exports" / EXPERIMENT
export_dir.mkdir(parents=True, exist_ok=True)
(export_dir / "m5_inference.py").write_text(INFERENCE_SOURCE, encoding="utf-8")
inference_config = {"model": PROTOCOL["model"], "imgsz": PROTOCOL["imgsz"],
    "threshold": chosen_threshold, "nms_iou": PROTOCOL["nms_iou"], "max_det": PROTOCOL["max_det"],
    "resize": "Square resize to 640; image INTER_AREA; masks INTER_NEAREST",
    "m5_class_id": 0, "m4_class_id": 3, "damage_code": "D40",
    "checkpoint_sha256": checkpoint_sha256, "protocol_hash": protocol_hash,
    "area_definition": "Union foreground pixels / all pixels of the supplied image",
    "limitations": "Area fraction is a visual proxy, not physical area or pothole depth. Vehicle-mounted/cropped input needs separate validation."}
save_json(export_dir / "inference_config.json", inference_config)
shutil.copy2(best_path, export_dir / "best.pt")
shutil.copy2(protocol_path, export_dir / "protocol.json")
shutil.copy2(selection_path, export_dir / "selection.json")
environment = {"python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "packages": {name: md.version(name) for name in ("ultralytics", "pycocotools", "wandb", "numpy", "opencv-python", "Pillow", "pandas", "matplotlib")},
    "seed": PROTOCOL["seed"], "note": "One training seed. Determinism is requested but hardware/library differences may remain."}
save_json(export_dir / "environment.json", environment)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(export_dir / "environment-full.txt").write_text(freeze, encoding="utf-8")
requirements = "\n".join(f"{p}=={md.version(p)}" for p in
    ("ultralytics", "pycocotools", "wandb", "numpy", "opencv-python", "Pillow", "pandas", "matplotlib", "PyYAML", "ipywidgets"))
(export_dir / "requirements.txt").write_text(
    "# Install a hardware-compatible torch/torchvision pair first; see environment.json.\n" + requirements + "\n", encoding="utf-8")
handoff = '''# M5 handoff

Load `best.pt` and `inference_config.json` with `m5_inference.py`:

```python
from m5_inference import PotholeSegmenter
segmenter = PotholeSegmenter("best.pt", "inference_config.json", device="cpu")
result = segmenter.predict("example.jpg")
print(result["area_percent"])
```

`mask` is a boolean array at the supplied image's original resolution.
`instances` contains masks, confidence and the mapping M5 class 0 -> M4 class 3 / D40.
Area uses the union of masks, so overlapping instances are not counted twice.
The notebook evaluates at 640 x 640; the helper also returns a mask resized to the caller's original image size.
For a crop, area_percent describes that crop, not the full road image. Mapping a crop mask back to a frame
requires the crop coordinates. This notebook does not claim M4-to-M5 integration has been tested.
The source videos are top-down pothole views; vehicle-mounted road images can have a domain gap.

No reference masks, ground-truth boxes or SAM prompts are used by this inference helper.
SAM in the notebook is a separate training-annotation review exercise.
HTTP serving, ONNX conversion and service latency belong to the later integration/deployment workstream.
Raw videos and extracted dataset frames are excluded from the submission package.
'''
(export_dir / "HANDOFF.md").write_text(handoff, encoding="utf-8")
print("Inference handoff saved:", export_dir)


## 17. Results Report

We will generate a report from the measured results. It will summarize the method, evaluation scores, area errors, failure analysis, and any review items that remain unfinished.

In [ ]:
sam_review = read_json(review_path) if review_path.exists() else {"complete": False}
sam_review_complete = bool(sam_review.get("complete"))
has_test = test_summary is not None
score_rows = []
for summary in [validation_summary] + ([test_summary] if has_test else []):
    operating = next(r for r in summary["thresholds"] if r["threshold"] == chosen_threshold)
    score_rows.append({"split": summary["split"], **summary["ap"], **operating})
comparison = pd.DataFrame(score_rows)
comparison.to_csv(export_dir / "metrics_summary.csv", index=False)
display(comparison)

checklist = [
    {"requirement": "Paired RGB/masks, clip-level split and technical audits", "status": "COMPLETE",
     "evidence": "video_manifest.csv; integrity/decode/similarity audit JSON files"},
    {"requirement": "SAM-generated proposals and visual verification", "status": "COMPLETE" if sam_review_complete else "PENDING VISUAL REVIEW",
     "evidence": "sam_review/proposals.json; visual_review.json; six comparison figures"},
    {"requirement": "Fine-tuned segmentation model", "status": "COMPLETE", "evidence": "best.pt; training/results.csv"},
    {"requirement": "Validation-only selection; final test metrics", "status": "COMPLETE" if has_test else "PENDING TEST",
     "evidence": "selection.json; test_lock.json; metrics_summary.csv"},
    {"requirement": "Mask mAP50-95, mean IoU and Dice", "status": "COMPLETE" if has_test else "VALIDATION ONLY",
     "evidence": "Reference-raster evaluation; connected-region AP caveat is documented"},
    {"requirement": "Area-based severity error", "status": "COMPLETE" if has_test else "VALIDATION ONLY",
     "evidence": "Severity MAE/RMSE/bias in percentage points; area scatter plot"},
    {"requirement": "At least one failure case and analysis", "status": "COMPLETE" if any("missed foreground pixels" in a for a in failure_analyses.values()) else "REVIEW NEEDED",
     "evidence": "failure_analyses.json; qualitative galleries"},
    {"requirement": "Model, configuration and reproducibility handoff", "status": "COMPLETE", "evidence": "exports folder"},
    {"requirement": "Cross-check any additional M5 lab-specific deliverables", "status": "NOT VERIFIED",
     "evidence": "The exact M5 laboratory handout was not available when this notebook was prepared."},
]
write_csv(export_dir / "completion_checklist.csv", checklist)
display(pd.DataFrame(checklist)[["requirement", "status"]])

metric_lines = ["| Split | Frames | Videos | Mask mAP50-95 | Mean IoU | FG IoU | Dice | Area MAE (pp) | Area RMSE (pp) |",
                "|---|---:|---:|---:|---:|---:|---:|---:|---:|"]
for r in score_rows:
    metric_lines.append(f"| {r['split']} | {r['frames']} | {r['videos']} | {r['mask_map50_95']:.4f} | {r['mean_iou']:.4f} | "
        f"{r['foreground_iou']:.4f} | {r['dice']:.4f} | {r['severity_mae_pp']:.3f} | {r['severity_rmse_pp']:.3f} |")
sam_status = "Completed and recorded" if sam_review_complete else "Pending: inspect all six proposals, record decisions/notes and save the visual review"
report = f'''# M5 Pothole Segmentation Report

**Contributor:** Abdulrahman Alsalmi, 225493
**Experiment:** {EXPERIMENT}
**Prepared from actual execution:** {datetime.now(timezone.utc).isoformat()}

## Task and scope
M5 segments pothole pixels, evaluates the masks and measures error in an area-based severity proxy.
The project guidelines require SAM-generated/refined and visually verified masks, a fine-tuned segmentation
model, mask mAP50-95, mean IoU, Dice and at least one analyzed failure. This work does not change M4.
The exact M5 lab handout was unavailable; additional lab-specific deliverables still need a cross-check.

## Data and preparation
Pothole Videos Dataset version 3 contains 619 RGB/mask pairs: 372 train, 124 validation and 123 test.
All 48 frames of every pair underwent technical decoding checks. The modeling benchmark uses the fixed
12 indices {PROTOCOL['frame_indices']}: 4,464 training, 1,488 validation and 1,476 test frames.
These are sampled-frame results, not an evaluation of all 29,712 source frames.
All frames from one clip stay in its published split. ID, encoded-file and representative-frame checks
are recorded; the similarity screen cannot rule out every repeated pothole identity or location.
Reference foreground is grayscale > 127 before nearest-neighbor resizing to 640 x 640. RGB uses area interpolation.
Raw videos/reference annotations were not edited. The original compressed masks and resizing introduce limitations.

## SAM annotation exercise
SAM-b generated six proposals on training frame 24 using boxes derived from training reference masks.
This is assisted annotation, not an automatic detection test. Proposals are compared with the RGB image
and reference, then visually reviewed. Original reference masks are retained; no unverified proposal is
silently used as ground truth. Review status: **{sam_status}**.
SAM/reference overlap on these prompted training examples is not model test performance.

## Model and selection
YOLO11n-seg was fine-tuned for up to {PROTOCOL['epochs']} epochs with batch {PROTOCOL['batch']}, input 640 x 640,
seed 42, AdamW and recorded augmentation settings. The pretrained model is adapted to one class: pothole.
The best checkpoint is selected by Ultralytics validation fitness (box mAP50-95 + mask mAP50-95 in the pinned version).
Those internal training metrics use polygon targets. Final reported mask metrics below use the reference binary rasters.
Operating confidence was selected from validation by maximum mean foreground Dice (ties prefer a higher threshold):
**{chosen_threshold:.2f}**. NMS IoU is {PROTOCOL['nms_iou']}; inference has at most {PROTOCOL['max_det']} predictions per image.
AP uses confidence floor {PROTOCOL['eval_conf_floor']} and IoU thresholds 0.50:0.05:0.95, independently of the operating threshold.
Checkpoint fingerprint: `{checkpoint_sha256}`. The protocol/selection are locked before test inference.

## Results and definitions
{chr(10).join(metric_lines)}

- **Mean IoU:** per-image average of foreground IoU and background IoU, then averaged over frames.
- **Foreground IoU and Dice:** foreground overlap scores, averaged over frames. Every video contributes 12 frames.
- **Mask AP:** COCO segmentation AP against 8-connected reference regions, without area filtering.
  The dataset provides semantic masks, not native instance IDs. This is a derived-region AP evaluation;
  disconnected pieces can be counted separately, and it is not directly interchangeable with a native-instance benchmark.
- **Severity proxy:** foreground union area / total image area, expressed as a percentage.
  MAE/RMSE compare predicted and reference percentages, so errors are in **percentage points**.
  This is not measured road-surface area, pothole depth or a validated engineering severity category.

## Failure analysis
{chr(10).join('### ' + s + chr(10) + a + chr(10) for s, a in failure_analyses.items())}

## Limitations and next experiments
This is one seed and a fixed sampled-frame benchmark. Nearby frames are correlated, and splitting by video
does not prove independence by physical pothole/location. The dataset uses top-down views from one region;
generalization to M4 vehicle-mounted road images remains untested. Binary semantic labels do not provide
native instance identities, and polygon conversion can change holes/boundaries. The conversion audit is saved.
Possible future improvements must use training/validation evidence. Test results are for reporting only.
No numerical minimum score is promised; performance must be assessed from the executed results.

## Handoff and references
The package contains the checkpoint, inference helper, fixed threshold, environment, manifests, metrics,
review records and figures. M5 class 0 maps to M4 D40/class 3. See HANDOFF.md for the inference contract.
Raw dataset videos/frames are excluded. Share source data under its license and attribution requirements.

- Project guidelines: Capstone_Project_Guidelines_UPDATED.pdf, M5 requirements (pages 6-7 and 32-33).
- [Pothole Videos Dataset v3](https://data.mendeley.com/datasets/5bwfg4v4cd/3), Ihsan, Harjoko and Amrizal, CC BY 4.0.
- [YOLO11](https://docs.ultralytics.com/models/yolo11/), [segmentation](https://docs.ultralytics.com/tasks/segment/),
  [SAM](https://docs.ultralytics.com/models/sam/), [COCO evaluation implementation](https://github.com/cocodataset/cocoapi).
'''
(export_dir / "M5_Report.md").write_text(report, encoding="utf-8")
print("Report saved:", export_dir / "M5_Report.md")
print("SAM review:", sam_status)


## 18. Experiment Tracking

We will record the training history, evaluation results, settings, and model artifact in Weights & Biases. This will make the experiment easier to review and share with the team.

In [ ]:
import wandb

verified = wandb.login(
    host="https://api.wandb.ai",
    relogin=True,
    verify=True,
)

print("Login verified:", verified)

In [ ]:
# Upload saved results in a fresh process.
# Training and evaluation are not repeated.

import json
import os
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd
import wandb

if WANDB_MODE not in ("online", "offline"):
    raise ValueError("WANDB_MODE must be online or offline.")

tracking_path = run_dir / "wandb_tracking.json"
old_tracking = read_json(tracking_path) if tracking_path.exists() else {}
run_id = old_tracking.get("run_id") or wandb.util.generate_id()

# Start without the previous W&B service settings.
upload_env = {
    key: value
    for key, value in os.environ.items()
    if not key.startswith(("WANDB_", "_WANDB_"))
}

# Reuse the credential from the successful login.
if WANDB_MODE == "online":
    from wandb.sdk.lib.apikey import api_key

    saved_key = api_key() or os.environ.get("WANDB_API_KEY")

    if not saved_key:
        raise RuntimeError(
            "Log in with wandb.login(relogin=True, verify=True), "
            "then rerun this cell."
        )

    upload_env["WANDB_API_KEY"] = saved_key
    del saved_key

# Copy saved outputs to local storage for uploading.
local_root = Path(tempfile.mkdtemp(prefix="m5_wb_"))

files = [
    "best.pt",
    "inference_config.json",
    "m5_inference.py",
    "protocol.json",
    "selection.json",
    "environment.json",
    "M5_Report.md",
    "metrics_summary.csv",
]

for name in files:
    shutil.copy2(export_dir / name, local_root / name)

shutil.copy2(
    training_dir / "results.csv",
    local_root / "training_history.csv",
)

review_path = run_dir / "sam_review" / "visual_review.json"
review_complete = (
    bool(read_json(review_path).get("complete"))
    if review_path.exists()
    else False
)

payload = {
    "entity": WANDB_ENTITY,
    "project": WANDB_PROJECT,
    "mode": WANDB_MODE,
    "name": EXPERIMENT,
    "run_id": run_id,
    "sam_review_complete": review_complete,
    "files": files + ["training_history.csv"],
}

(local_root / "payload.json").write_text(
    json.dumps(payload),
    encoding="utf-8",
)

for variable, folder in [
    ("WANDB_DIR", "logs"),
    ("WANDB_CACHE_DIR", "cache"),
    ("WANDB_DATA_DIR", "data"),
    ("WANDB_CONFIG_DIR", "config"),
]:
    location = local_root / folder
    location.mkdir()
    upload_env[variable] = str(location)

upload_env.update(
    WANDB_BASE_URL="https://api.wandb.ai",
    WANDB_CONSOLE="off",
)

# This script runs separately from the notebook's W&B service.
worker_code = r'''
import hashlib
import json
import os
from pathlib import Path

import pandas as pd
import requests
import wandb

root = Path(__file__).resolve().parent
p = json.loads(
    (root / "payload.json").read_text(encoding="utf-8")
)

def phase(message):
    (root / "phase.txt").write_text(
        message,
        encoding="utf-8",
    )

phase("Checking saved model")

selection = json.loads(
    (root / "selection.json").read_text(encoding="utf-8")
)

digest = hashlib.sha256(
    (root / "best.pt").read_bytes()
).hexdigest()

if digest != selection["checkpoint_sha256"]:
    raise RuntimeError(
        "Saved model does not match the selected checkpoint. "
        "Upload stopped."
    )

if p["mode"] == "online":
    phase("Checking account and project access")

    query = """
    query($e: String!, $p: String!) {
        viewer {
            username
        }
        project(name: $p, entityName: $e) {
            name
        }
    }
    """

    response = requests.post(
        "https://api.wandb.ai/graphql",
        auth=("api", os.environ["WANDB_API_KEY"]),
        json={
            "query": query,
            "variables": {
                "e": p["entity"],
                "p": p["project"],
            },
        },
        timeout=(10, 20),
        allow_redirects=False,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"W&B access check returned HTTP {response.status_code}."
        )

    reply = response.json()
    data = reply.get("data") or {}

    if (
        reply.get("errors")
        or not data.get("viewer")
        or not data.get("project")
    ):
        raise RuntimeError(
            "Account/project access check failed. "
            "Check entity, project and permissions."
        )

config = json.loads(
    (root / "protocol.json").read_text(encoding="utf-8")
)

config.update(
    selected_threshold=selection["threshold"],
    checkpoint_sha256=digest,
)

phase("Starting a fresh W&B service and run")

run = wandb.init(
    entity=p["entity"],
    project=p["project"],
    id=p["run_id"],
    resume="allow" if p["mode"] == "online" else None,
    mode=p["mode"],
    name=p["name"],
    job_type="segmentation",
    tags=["M5", "pothole", "YOLO11n-seg"],
    config=config,
    dir=str(root / "logs"),
    settings=wandb.Settings(init_timeout=120),
)

phase("Uploading saved history and metrics")

history = pd.read_csv(root / "training_history.csv")
history.columns = history.columns.str.strip()
next_step = run.step

for record in history.to_dict("records"):
    epoch = int(record["epoch"])

    if epoch >= next_step:
        values = {
            f"training/{key}": float(value)
            for key, value in record.items()
            if key != "epoch" and pd.notna(value)
        }

        run.log(
            {"epoch": epoch, **values},
            step=epoch,
        )

comparison = pd.read_csv(root / "metrics_summary.csv")

for record in comparison.to_dict("records"):
    for key, value in record.items():
        if key != "split":
            run.summary[f"{record['split']}/{key}"] = value

run.summary["sam_visual_review_complete"] = (
    p["sam_review_complete"]
)

run.log({
    "metrics_comparison": wandb.Table(dataframe=comparison)
})

phase("Uploading model artifact")

artifact = wandb.Artifact(
    p["name"] + "-model",
    type="model",
    metadata={"checkpoint_sha256": digest},
)

for name in p["files"]:
    artifact.add_file(
        str(root / name),
        name=name,
    )

logged = run.log_artifact(artifact)

if p["mode"] == "online":
    logged.wait(timeout=120)

info = {
    "run_id": p["run_id"],
    "mode": p["mode"],
    "history_logged": True,
    "status": (
        "SYNCED"
        if p["mode"] == "online"
        else "OFFLINE_PENDING_SYNC"
    ),
    "url": run.url if p["mode"] == "online" else None,
    "sync_directory": str(Path(run.dir).parent),
    "collaborator_access": (
        "Not verified; a teammate must check access "
        "using their own account."
    ),
}

phase("Finishing synchronization")
run.finish()

(root / "result.json").write_text(
    json.dumps(info),
    encoding="utf-8",
)

phase("Finished")
'''

(local_root / "upload.py").write_text(
    worker_code,
    encoding="utf-8",
)

save_json(
    tracking_path,
    {
        **old_tracking,
        "run_id": run_id,
        "mode": WANDB_MODE,
        "status": "STARTED",
    },
)

print(
    "Uploading saved files only. "
    "Progress appears every 30 seconds.",
    flush=True,
)

process = subprocess.Popen(
    [sys.executable, "-u", str(local_root / "upload.py")],
    cwd=str(local_root),
    env=upload_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    start_new_session=(os.name == "posix"),
)

started = time.monotonic()

try:
    while True:
        try:
            output, _ = process.communicate(timeout=30)
            break

        except subprocess.TimeoutExpired:
            phase_path = local_root / "phase.txt"

            stage = (
                phase_path.read_text(encoding="utf-8")
                if phase_path.exists()
                else "Starting upload"
            )

            elapsed = int(time.monotonic() - started)
            print(f"{elapsed}s: {stage}", flush=True)

            if elapsed >= 420:
                raise TimeoutError(
                    "Upload exceeded seven minutes. "
                    "Saved results are unchanged."
                )

except BaseException:
    if os.name == "posix":
        os.killpg(process.pid, signal.SIGTERM)
    else:
        process.kill()

    save_json(
        tracking_path,
        {
            **old_tracking,
            "run_id": run_id,
            "mode": WANDB_MODE,
            "status": "UPLOAD_INTERRUPTED",
            "local_directory": str(local_root),
        },
    )

    upload_env.pop("WANDB_API_KEY", None)
    raise

# Save diagnostic messages without the API key.
secret = upload_env.pop("WANDB_API_KEY", None)

if secret:
    output = output.replace(secret, "[REDACTED]")

del secret

output = re.sub(
    r"wandb_v1_\S+",
    "[REDACTED]",
    output,
)

(run_dir / "wandb_upload_attempt.log").write_text(
    output,
    encoding="utf-8",
)

if process.returncode != 0 or not (local_root / "result.json").exists():
    save_json(
        tracking_path,
        {
            **old_tracking,
            "run_id": run_id,
            "mode": WANDB_MODE,
            "status": "UPLOAD_FAILED",
            "local_directory": str(local_root),
        },
    )

    print("\n".join(output.splitlines()[-12:]))

    raise RuntimeError(
        "Upload failed; saved training and test results "
        "are unchanged. Share the final error above."
    )

tracking_info = read_json(local_root / "result.json")

if WANDB_MODE == "offline":
    # Preserve offline data on Drive.
    backup = run_dir / "offline_uploads" / local_root.name

    shutil.copytree(
        tracking_info["sync_directory"],
        backup,
    )

    tracking_info["sync_directory"] = str(backup)

save_json(tracking_path, tracking_info)

save_json(
    export_dir / "wandb_tracking.json",
    tracking_info,
)

run_url = tracking_info["url"]

tracking_row = {
    "requirement": "W&B experiment and model tracking",
    "status": "COMPLETE" if run_url else "PENDING ONLINE SYNC",
    "evidence": run_url or tracking_info["sync_directory"],
}

checklist = (
    pd.read_csv(export_dir / "completion_checklist.csv")
    .fillna("")
    .to_dict("records")
)

checklist = [
    row
    for row in checklist
    if row["requirement"] != tracking_row["requirement"]
] + [tracking_row]

write_csv(
    export_dir / "completion_checklist.csv",
    checklist,
)

report_path = export_dir / "M5_Report.md"

body = (
    report_path.read_text(encoding="utf-8")
    .split("\n## Experiment tracking\n")[0]
)

status_text = (
    f"Recorded online: {run_url}"
    if run_url
    else "Offline run saved; online synchronization is still pending."
)

report_path.write_text(
    body
    + "\n## Experiment tracking\n"
    + status_text
    + "\nCollaborator access has not been verified. "
      "A teammate must check the run and model artifact.\n",
    encoding="utf-8",
)

if run_url:
    print("SUCCESS — W&B run:", run_url)
else:
    print(
        "Offline data saved:",
        tracking_info["sync_directory"],
    )

## 19. Saving the Results

Finally, we will collect the model, reports, figures, review records, and supporting files into a ZIP file in Google Drive. The raw videos and extracted dataset frames will remain outside this package.

In [ ]:
# Copy only reproducibility evidence, model outputs and small review figures into the handoff folder.
for source_dir, target_name in [(figure_dir, "figures"), (sam_dir, "sam_review")]:
    shutil.copytree(source_dir, export_dir / target_name, dirs_exist_ok=True)
shutil.copy2(run_dir / "learning_curves.png", export_dir / "figures/learning_curves.png")
shutil.copy2(run_dir / "polygon_label_preview.png", export_dir / "figures/polygon_label_preview.png")
shutil.copy2(training_dir / "results.csv", export_dir / "training_history.csv")
shutil.copy2(training_dir / "args.yaml", export_dir / "training_args.yaml")
shutil.copy2(run_dir / "polygon_conversion_audit.json", export_dir / "polygon_conversion_audit.json")
shutil.copy2(result_dir / "failure_analyses.json", export_dir / "failure_analyses.json")
if test_lock_path.exists():
    shutil.copy2(test_lock_path, export_dir / "test_lock.json")
for name in ("archive_pairing_audit.json", "video_file_integrity_audit.json", "video_decode_audit.json", "visual_duplicate_screening.json"):
    target = export_dir / "audits" / name
    target.parent.mkdir(exist_ok=True)
    shutil.copy2(m5_dir / "reports" / name, target)
for path in [m5_dir / "data/video_manifest.csv", m5_dir / "data/video_file_fingerprints.csv",
             m5_dir / "data/video_decode_metadata.csv", frame_manifest_path]:
    target = export_dir / "manifests" / path.name
    target.parent.mkdir(exist_ok=True)
    shutil.copy2(path, target)
for split in ("val", "test"):
    source = result_dir / split
    if not (source / "summary.json").exists():
        continue
    target = export_dir / "metrics" / split
    target.mkdir(parents=True, exist_ok=True)
    for path in source.iterdir():
        if path.is_file() and (path.suffix == ".csv" or path.name in ("summary.json", "coco_evaluation.txt")):
            shutil.copy2(path, target / path.name)
delivery_files = [{"file": str(p.relative_to(export_dir)).replace("\\", "/"), "bytes": p.stat().st_size,
                   "sha256": sha256_file(p)} for p in sorted(export_dir.rglob("*"))
                  if p.is_file() and p.name != "artifact_manifest.json"]
save_json(export_dir / "artifact_manifest.json", delivery_files)
package_path = m5_dir / "exports" / f"{EXPERIMENT}_submission.zip"
temporary_package = package_path.with_suffix(".zip.part")
with zipfile.ZipFile(temporary_package, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(export_dir.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(Path(EXPERIMENT) / path.relative_to(export_dir)))
temporary_package.replace(package_path)
with zipfile.ZipFile(package_path) as archive:
    corrupt = archive.testzip()
    if corrupt:
        raise RuntimeError(f"Package verification failed: {corrupt}")
print("Package saved:", package_path)
print("Package size (MiB):", round(package_path.stat().st_size / 1024**2, 1))
print("Save this notebook with its outputs using File > Save, then File > Download > Download .ipynb.")
print("Keep the name M5_Pothole_Segmentation.ipynb. Return the executed notebook and metrics_summary.csv for review.")
print("If SAM review is still pending, finish it and rerun the report/package cells; no retraining is needed.")


## Experiment Summary

| Item | Method |
|---|---|
| Sampled frames | 4,464 training, 1,488 validation, 1,476 test |
| Image and mask size | 640 x 640 |
| Reference foreground | Grayscale value greater than 127 |
| Mask resizing | Nearest-neighbor interpolation |
| Checkpoint selection | Combined box and mask validation fitness |
| Confidence selection | Highest mean foreground Dice on validation |
| Mask AP | IoU thresholds 0.50–0.95; prediction confidence floor 0.001 |
| Mean IoU | Foreground/background IoU averaged per image, then across images |
| Dice | Foreground Dice averaged across images |
| Area estimate | Predicted foreground union as a percentage of image area |
| Area error | MAE, RMSE, and signed bias in percentage points |

These results will describe the sampled frames, rather than all source frames. The area estimate is an image-based severity proxy, not a measurement of physical area or pothole depth. Connected regions provide a derived mask-AP evaluation because the source dataset has no native instance IDs. Performance on M4 vehicle-mounted images will require separate validation.

## References

- Course project brief: Capstone_Project_Guidelines_UPDATED.pdf, pages 6–7 and 32–33.
- [Pothole Videos Dataset v3](https://data.mendeley.com/datasets/5bwfg4v4cd/3), Ihsan, Harjoko and Amrizal, CC BY 4.0.
- [YOLO11](https://docs.ultralytics.com/models/yolo11/), [segmentation](https://docs.ultralytics.com/tasks/segment/), and [SAM](https://docs.ultralytics.com/models/sam/).
- [COCO evaluation](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py).

# Additional M5 Laboratory Analyses

We will adapt the relevant Day 6 instance-segmentation exercises to our pothole project. We will keep the original
executed cells, outputs, trained model, selected confidence threshold and primary test results. Start at **Section 20**
and run the new sections in order on a Colab GPU; the original training does not need to run again.

We will compare actual detector-box and mask AP, object-level precision/recall/F1, and SAM masks prompted by the
detector's boxes. A reconstructed initialization provides a validation-only baseline. The original starting weights
were not saved, so this baseline will be described as a reconstruction, not an exact historical checkpoint.

These analyses are supplementary and were added after the original test evaluation. We will not use them to replace
the selected model or threshold. Architecture-specific lab exercises will be adapted and documented rather than
presented as an exact reproduction of the COCO/Mask R-CNN classroom experiment. A new ZIP will include all results,
while retaining the previous submission package.


## 20. Reconnecting to the Saved Experiment

We will reconnect to Google Drive, load the saved experiment settings and verify the selected model. We will also fix the 100-video validation subset for the additional comparisons.


In [ ]:
# Reconnect to the saved experiment; no training is started here.
import importlib.metadata as lab_md
import importlib.util as lab_import
import subprocess as lab_subprocess
import sys as lab_sys

lab_required = {'ultralytics': '8.4.197', 'pycocotools': '2.0.10', 'wandb': '0.22.3'}
lab_install = []
for lab_name, lab_version in lab_required.items():
    try:
        if lab_md.version(lab_name) != lab_version:
            lab_install.append(f'{lab_name}=={lab_version}')
    except lab_md.PackageNotFoundError:
        lab_install.append(f'{lab_name}=={lab_version}')
if lab_install:
    lab_subprocess.check_call([lab_sys.executable, '-m', 'pip', 'install', '-q', *lab_install])

import copy
import gc
import hashlib
import io
import json
import os
import random
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from IPython.display import display, Markdown
from pycocotools import mask as lab_mask_util
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO, settings
settings.update({'wandb': False, 'mlflow': False, 'comet': False})

def lab_read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def lab_save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.part')
    temp.write_text(json.dumps(value, allow_nan=False, separators=(',', ':')), encoding='utf-8')
    temp.replace(path)

def lab_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(4 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def lab_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

from google.colab import drive
drive.mount('/content/drive')
lab_projects = [p for p in Path('/content/drive/MyDrive').iterdir()
                if p.is_dir() and p.name.strip() == 'AI Project']
if len(lab_projects) != 1:
    raise RuntimeError('Add the AI Project folder shortcut to MyDrive, then rerun this section.')
lab_project = lab_projects[0]
lab_m5 = lab_project / 'M5_Segmentation'
lab_experiment = 'm5_yolo11n_seg_v1'
lab_export = lab_m5 / 'exports' / lab_experiment
lab_run = lab_m5 / 'runs' / lab_experiment
lab_dir = lab_run / 'lab_extension_v1'
lab_dir.mkdir(parents=True, exist_ok=True)
lab_protocol = lab_read(lab_export / 'protocol.json')
lab_selection = lab_read(lab_export / 'selection.json')
lab_lock = lab_read(lab_export / 'test_lock.json')
lab_checkpoint = lab_export / 'best.pt'
lab_threshold = float(lab_selection['threshold'])
lab_device = '0' if torch.cuda.is_available() else 'cpu'
if lab_device == 'cpu':
    raise RuntimeError('Select a Colab GPU before running these additional comparisons. Saved results are unchanged.')
if lab_sha(lab_checkpoint) != lab_selection['checkpoint_sha256'] or lab_hash(lab_selection) != lab_lock['selection_fingerprint']:
    raise RuntimeError('The selected checkpoint or locked settings have changed. Stop and review the saved experiment.')
if lab_hash(lab_protocol) != lab_selection['protocol_hash']:
    raise RuntimeError('The saved protocol differs from the selected experiment.')
lab_save(lab_dir / 'environment_supplement.json', {
    'python': lab_sys.version, 'torch': str(torch.__version__),
    'cuda': torch.version.cuda, 'device': torch.cuda.get_device_name(0),
    'packages': {name: lab_md.version(name) for name in
                 ['ultralytics', 'pycocotools', 'wandb', 'numpy', 'opencv-python', 'Pillow', 'pandas', 'matplotlib']}})
lab_protected = ['best.pt', 'protocol.json', 'selection.json', 'test_lock.json',
                 'inference_config.json', 'metrics_summary.csv']
lab_original_hashes = {name: lab_sha(lab_export / name) for name in lab_protected}
lab_spec = {'version': 1, 'protocol_hash': lab_hash(lab_protocol),
            'checkpoint_sha256': lab_selection['checkpoint_sha256'], 'operating_threshold': lab_threshold,
            'comparison_subset': 'first 100 sorted validation video IDs, sampled frame 26',
            'comparison_frame': 26, 'comparison_videos': 100,
            'ap_confidence_floor': lab_protocol['eval_conf_floor'], 'nms_iou': lab_protocol['nms_iou'],
            'max_detections': lab_protocol['max_det'], 'resolution': lab_protocol['imgsz'],
            'object_matching_iou': 0.5, 'original_hashes': lab_original_hashes,
            'scope': 'Post-hoc supplementary analysis. Original model, operating threshold and test results remain fixed.'}
lab_spec_path = lab_dir / 'extension_protocol.json'
if lab_spec_path.exists() and lab_read(lab_spec_path) != lab_spec:
    raise RuntimeError('An existing supplement has different settings. Do not mix experiments.')
lab_save(lab_spec_path, lab_spec)
lab_manifest_path = lab_export / 'manifests' / f"frame_manifest_{lab_selection['protocol_hash'][:12]}.csv"
lab_frames = pd.read_csv(lab_manifest_path, dtype={'video_id': str}).sort_values(['split', 'video_id', 'frame_index'])
lab_subset = lab_frames[(lab_frames['split'] == 'val') & (lab_frames['frame_index'] == 26)].head(100).to_dict('records')
if len(lab_subset) != 100 or len({r['video_id'] for r in lab_subset}) != 100:
    raise RuntimeError('Expected 100 different validation videos for the comparison.')
lab_save(lab_dir / 'comparison_subset.json', [{'video_id': r['video_id'], 'frame_index': int(r['frame_index'])} for r in lab_subset])
lab_local = Path('/content/m5_lab_extension')
lab_local.mkdir(exist_ok=True)
lab_frame_dir = Path('/content/m5_local') / f"frames_{lab_selection['protocol_hash'][:12]}"
print('Saved model:', lab_checkpoint)
print('Fixed operating threshold:', lab_threshold)
print('Supplement:', lab_dir)
print('GPU:', torch.cuda.get_device_name(0))


## 21. Objects, Polygons and RLE

We will summarize the training regions by size and measure how polygon simplification changes their masks. We will also check a manual RLE round trip against pycocotools. Missing sampled frames will be restored from the existing archive.


In [ ]:
# Recover only missing sampled frames, using the original manifest and archive.
lab_verified_paths = set()

def lab_ensure_frames(records, include_rgb=True):
    groups = defaultdict(list)
    for row in records:
        groups[(row['split'], row['video_id'])].append(row)
    archive = None
    for number, ((split, video_id), group) in enumerate(groups.items(), 1):
        for kind, field in [('mask', 'mask_path')] + ([('rgb', 'image_path')] if include_rgb else []):
            missing = []
            for row in group:
                target = lab_frame_dir / row[field]
                if str(target) not in lab_verified_paths:
                    expected = row['mask_sha256' if kind == 'mask' else 'image_sha256']
                    if target.exists() and lab_sha(target) == expected:
                        lab_verified_paths.add(str(target))
                    else:
                        missing.append(row)
            if not missing:
                continue
            source = Path('/content/m5_local/videos') / split / kind / f'{video_id}.mp4'
            if not source.is_file():
                if archive is None:
                    local_zip = lab_local / 'pothole_video.zip'
                    if not local_zip.exists() or lab_sha(local_zip) != lab_protocol['archive_sha256']:
                        print('Restoring missing frames from the saved dataset archive...', flush=True)
                        shutil.copy2(lab_project / 'pothole_video.zip', local_zip)
                        if lab_sha(local_zip) != lab_protocol['archive_sha256']:
                            raise RuntimeError('Dataset archive fingerprint differs from the original experiment.')
                    archive = zipfile.ZipFile(local_zip)
                source = lab_local / 'videos' / split / kind / f'{video_id}.mp4'
                source.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(f'pothole_video/{split}/{kind}/{video_id}.mp4') as src, source.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
            needed = {int(row['frame_index']): row for row in missing}
            cap = cv2.VideoCapture(str(source))
            try:
                for frame_index in range(max(needed) + 1):
                    ok, frame = cap.read()
                    if not ok:
                        raise RuntimeError(f'Cannot decode {split}/{video_id}/{frame_index}.')
                    if frame_index not in needed:
                        continue
                    row = needed[frame_index]
                    size = int(lab_protocol['imgsz'])
                    if kind == 'mask':
                        frame = (cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) > lab_protocol['mask_threshold']).astype(np.uint8)
                        frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_NEAREST) * 255
                    else:
                        frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)
                    target = lab_frame_dir / row[field]
                    target.parent.mkdir(parents=True, exist_ok=True)
                    if not cv2.imwrite(str(target), frame):
                        raise IOError(f'Cannot save {target}.')
                    expected = row['mask_sha256' if kind == 'mask' else 'image_sha256']
                    if lab_sha(target) != expected:
                        raise RuntimeError('Recovered frame differs from the saved manifest; do not mix decoding environments.')
                    lab_verified_paths.add(str(target))
            finally:
                cap.release()
        if number == 1 or number % 50 == 0 or number == len(groups):
            print(f'Frame availability checked: {number}/{len(groups)} videos', flush=True)
    if archive is not None:
        archive.close()

def lab_components(mask):
    count, labels = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
    return [labels == i for i in range(1, count)]

def lab_rle(mask):
    value = lab_mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))
    return {'size': list(value['size']), 'counts': value['counts'].decode('ascii')}

def lab_decode(value):
    return lab_mask_util.decode({'size': value['size'], 'counts': value['counts'].encode('ascii')}).astype(bool)

def lab_size(area):
    return 'small' if area < 32**2 else ('medium' if area < 96**2 else 'large')

def lab_iou(a, b):
    union = np.count_nonzero(a | b)
    return float(np.count_nonzero(a & b) / union) if union else 1.0

def lab_gt(records):
    gt = {'info': {'description': '8-connected regions from original binary masks; IDs are derived, not native instances.'},
          'images': [], 'annotations': [], 'categories': [{'id': 1, 'name': 'pothole'}]}
    for image_id, row in enumerate(records, 1):
        mask = cv2.imread(str(lab_frame_dir / row['mask_path']), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(row['mask_path'])
        gt['images'].append({'id': image_id, 'file_name': row['image_path'], 'height': mask.shape[0], 'width': mask.shape[1]})
        for component in lab_components(mask > 127):
            encoded = lab_rle(component)
            gt['annotations'].append({'id': len(gt['annotations']) + 1, 'image_id': image_id, 'category_id': 1,
                'segmentation': encoded, 'area': int(component.sum()),
                'bbox': [float(x) for x in lab_mask_util.toBbox(encoded)], 'iscrowd': 0})
    return gt

# Object counts and polygon simplification use training annotations only.
lab_train_rows = lab_frames[lab_frames['split'] == 'train'].to_dict('records')
lab_data_path = lab_dir / 'data_exercises.json'
if not lab_data_path.exists():
    lab_ensure_frames(lab_train_rows, include_rgb=False)
    lab_counts = Counter()
    lab_samples = Counter()
    lab_polygon_rows = []
    lab_example = None
    for row in lab_train_rows:
        mask = cv2.imread(str(lab_frame_dir / row['mask_path']), cv2.IMREAD_GRAYSCALE) > 127
        for component in lab_components(mask):
            area = int(component.sum())
            size = lab_size(area)
            lab_counts[size] += 1
            if lab_samples[size] >= 150:
                continue
            lab_samples[size] += 1
            if lab_example is None:
                lab_example = component
            contours, _ = cv2.findContours(component.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            for eps in (0, 1, 2, 4):
                recovered = np.zeros_like(component, np.uint8)
                for contour in contours:
                    polygon = cv2.approxPolyDP(contour, eps, True) if eps else contour
                    if len(polygon) >= 3:
                        cv2.fillPoly(recovered, [polygon], 1)
                lab_polygon_rows.append({'size': size, 'epsilon_pixels': eps, 'iou': lab_iou(component, recovered.astype(bool))})
    lab_polygon_table = pd.DataFrame(lab_polygon_rows).groupby(['size', 'epsilon_pixels'], sort=False).agg(
        objects=('iou', 'size'), mean_iou=('iou', 'mean'), minimum_iou=('iou', 'min')).reset_index()
    lab_polygon_table.to_csv(lab_dir / 'polygon_simplification.csv', index=False)
    # Uncompressed COCO RLE starts with the length of a zero run.
    def lab_manual_rle_encode(mask, order='F'):
        counts, previous, run = [], 0, 0
        for pixel in mask.astype(np.uint8).ravel(order=order):
            if pixel == previous:
                run += 1
            else:
                counts.append(run)
                previous, run = int(pixel), 1
        return counts + [run]
    def lab_manual_rle_decode(counts, shape, order='F'):
        flat = np.concatenate([np.full(n, i % 2, np.uint8) for i, n in enumerate(counts)])
        return flat.reshape(shape, order=order).astype(bool)
    encoded = lab_manual_rle_encode(lab_example)
    decoded = lab_manual_rle_decode(encoded, lab_example.shape)
    official = lab_mask_util.frPyObjects({'size': list(lab_example.shape), 'counts': encoded}, *lab_example.shape)
    assert np.array_equal(decoded, lab_example) and np.array_equal(lab_mask_util.decode(official).astype(bool), lab_example)
    wrong = lab_manual_rle_decode(lab_manual_rle_encode(lab_example, 'C'), lab_example.shape, 'F')
    lab_save(lab_data_path, {'training_frames': len(lab_train_rows), 'derived_objects': sum(lab_counts.values()),
        'class_counts': {'pothole': sum(lab_counts.values())}, 'size_counts': dict(lab_counts),
        'size_definition': 'Reference-region area in pixels at 640x640; small <1024, medium <9216, else large.',
        'crowd': 'The source supplies binary semantic masks and no crowd/instance metadata; all derived regions use iscrowd=0.',
        'manual_rle_matches_pycocotools': True, 'wrong_order_iou': lab_iou(lab_example, wrong),
        'polygon_note': 'External-contour simplification can lose holes or tiny components; the original reference masks stay unchanged.'})
display(pd.DataFrame([lab_read(lab_data_path)['size_counts']]))
display(pd.read_csv(lab_dir / 'polygon_simplification.csv'))
print('RLE verification:', lab_read(lab_data_path)['manual_rle_matches_pycocotools'])


## 22. Additional Evaluation Functions

We will prepare box and mask AP, one-to-one object matching and a manual AP50 check. Mask AP will use mask area, and object-level precision/recall will count regions rather than pixels.


In [ ]:
def lab_coco_ap(gt, predictions, kind='segm'):
    # Mask AP must use mask area, not the area of a detector box.
    key = 'segmentation' if kind == 'segm' else 'bbox'
    clean = [{k: p[k] for k in ('image_id', 'category_id', 'score', key)} for p in predictions]
    capture = io.StringIO()
    with redirect_stdout(capture):
        ground = COCO()
        ground.dataset = copy.deepcopy(gt)
        ground.createIndex()
        if clean:
            detected = ground.loadRes(clean)
        else:
            detected = COCO()
            detected.dataset = {'images': gt['images'], 'annotations': [], 'categories': gt['categories']}
            detected.createIndex()
        evaluator = COCOeval(ground, detected, kind)
        evaluator.params.imgIds = [r['id'] for r in gt['images']]
        evaluator.params.catIds = [1]
        evaluator.params.iouThrs = np.linspace(.5, .95, 10)
        evaluator.params.maxDets = [1, 10, 100]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    names = ['AP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']
    return {k: float(v) if v >= 0 else None for k, v in zip(names, evaluator.stats[:6])}, capture.getvalue()

def lab_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def lab_predictions(checkpoint, records, tag):
    checkpoint = Path(checkpoint)
    checkpoint_hash = lab_sha(checkpoint)
    directory = lab_dir / 'prediction_cache' / tag
    directory.mkdir(parents=True, exist_ok=True)
    model, predictions, times, devices = None, [], [], []
    for image_id, row in enumerate(records, 1):
        identity = lab_hash({'checkpoint': checkpoint_hash, 'image': row['image_sha256'],
                             'spec': lab_spec, 'tag': tag})
        path = directory / f"{row['split']}_{row['video_id']}_f{int(row['frame_index']):02d}.json"
        if path.exists():
            saved = lab_read(path)
            if saved['fingerprint'] != identity:
                raise RuntimeError('Cached predictions use different settings; stop rather than overwrite them.')
        else:
            if model is None:
                model = YOLO(str(checkpoint))
                if model.names != {0: 'pothole'}:
                    raise RuntimeError('The comparison requires a one-class pothole model.')
                # Warm up outside the recorded per-image timing.
                model.predict(np.zeros((640,640,3), np.uint8), imgsz=lab_protocol['imgsz'],
                              device=lab_device, verbose=False)
            lab_sync()
            started = time.perf_counter()
            result = model.predict(str(lab_frame_dir / row['image_path']), imgsz=lab_protocol['imgsz'],
                conf=lab_protocol['eval_conf_floor'], iou=lab_protocol['nms_iou'],
                max_det=lab_protocol['max_det'], retina_masks=True, device=lab_device,
                half=False, augment=False, verbose=False)[0]
            instances = []
            if result.masks is not None:
                masks = result.masks.data.cpu().numpy() > .5
                boxes, scores, classes = [x.cpu().numpy() for x in (result.boxes.xyxy, result.boxes.conf, result.boxes.cls)]
                if not (len(masks) == len(boxes) == len(scores)):
                    raise RuntimeError('Mask/box counts differ.')
                for mask, box, score, category in zip(masks, boxes, scores, classes):
                    if int(category) != 0 or mask.shape != (640,640):
                        raise RuntimeError('Unexpected class or mask size.')
                    if not mask.any():
                        continue
                    x1,y1,x2,y2 = map(float, box)
                    instances.append({'category_id': 1, 'score': float(score),
                        'bbox': [x1,y1,x2-x1,y2-y1], 'segmentation': lab_rle(mask)})
            lab_sync()
            saved = {'fingerprint': identity, 'seconds': time.perf_counter()-started, 'instances': instances,
                     'timing_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
                     'torch_version': str(torch.__version__)}
            lab_save(path, saved)
        predictions.extend({'image_id': image_id, **p} for p in saved['instances'])
        times.append(saved['seconds'])
        devices.append(saved.get('timing_device', 'not recorded'))
        if image_id == 1 or image_id % 100 == 0 or image_id == len(records):
            print(f'{tag}: {image_id}/{len(records)}', flush=True)
    if model is not None:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    lab_save(lab_dir / f'{tag}_timing.json', {
        'images': len(times), 'device_counts': dict(Counter(devices)),
        'mean_ms_per_image': 1000 * float(np.mean(times)) if times else None,
        'scope': 'Warm-up excluded; image loading, inference, postprocessing and RLE serialization included. Cached timings retain their original execution device.'})
    return predictions, times

def lab_match_mask_predictions(gt, predictions, confidence=0.0):
    annotations = defaultdict(list)
    detected = defaultdict(list)
    for g in gt['annotations']:
        if not g.get('iscrowd', 0):
            annotations[g['image_id']].append(g)
    for p in predictions:
        if p['score'] >= confidence:
            detected[p['image_id']].append(p)
    outcomes = []
    for image in gt['images']:
        objects = annotations[image['id']]
        current = sorted(detected[image['id']], key=lambda p: -p['score'])[:100]
        if objects and current:
            matrix = lab_mask_util.iou([p['segmentation'] for p in current],
                                      [g['segmentation'] for g in objects], [0]*len(objects))
        else:
            matrix = np.zeros((len(current), len(objects)))
        used = set()
        for i, prediction in enumerate(current):
            candidates = [(float(matrix[i,j]), j) for j,g in enumerate(objects)
                          if j not in used and g['category_id'] == prediction['category_id']]
            best = max(candidates, default=(-1.0,-1))
            matched = best[0] >= .5
            if matched:
                used.add(best[1])
            outcomes.append({'score': prediction['score'], 'tp': int(matched), 'fp': int(not matched)})
    return outcomes, sum(map(len, annotations.values()))

def lab_object_scores(gt, predictions, threshold):
    matched, count = lab_match_mask_predictions(gt, predictions, threshold)
    tp, fp = sum(p['tp'] for p in matched), sum(p['fp'] for p in matched)
    fn = count-tp
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/count if count else 0.0
    return {'threshold': float(threshold), 'tp': tp, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall,
            'f1': 2*precision*recall/(precision+recall) if precision+recall else 0.0}

def lab_manual_ap50(gt, predictions):
    matched, count = lab_match_mask_predictions(gt, predictions)
    if not count:
        return None
    if not matched:
        return 0.0
    matched = sorted(matched, key=lambda p: -p['score'])
    tp = np.cumsum([p['tp'] for p in matched])
    fp = np.cumsum([p['fp'] for p in matched])
    recall, precision = tp/count, tp/np.maximum(tp+fp, 1)
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    return float(np.mean([precision[recall >= x].max() if np.any(recall >= x) else 0.0
                          for x in np.linspace(0,1,101)]))

def lab_ap_rows(gt, predictions, label, split):
    output = []
    for kind in ('bbox', 'segm'):
        stats, log = lab_coco_ap(gt, predictions, kind)
        (lab_dir / f'{label}_{split}_{kind}_coco.txt').write_text(log, encoding='utf-8')
        output.append({'model': label, 'split': split, 'task': kind, **stats})
    return output

print('Supplementary box/mask AP and object-level matching functions are ready.')


## 23. Box and Mask AP by Object Size

We will collect actual predicted boxes and masks from the saved model and report AP overall and by size. This additional pass keeps the original settings fixed and checks agreement with the original mask AP; it does not replace the primary test results.


In [ ]:
# Collect actual detector boxes as well as masks from the same saved checkpoint.
lab_val_records = lab_frames[lab_frames['split'] == 'val'].to_dict('records')
lab_test_records = lab_frames[lab_frames['split'] == 'test'].to_dict('records')
lab_ensure_frames(lab_val_records + lab_test_records)
lab_val_gt = lab_gt(lab_val_records)
lab_test_gt = lab_gt(lab_test_records)
lab_val_predictions, lab_val_times = lab_predictions(lab_checkpoint, lab_val_records, 'finetuned_val')
lab_test_predictions, lab_test_times = lab_predictions(lab_checkpoint, lab_test_records, 'finetuned_test_report_only')
lab_ap_table = pd.DataFrame(lab_ap_rows(lab_val_gt, lab_val_predictions, 'finetuned', 'val') +
                          lab_ap_rows(lab_test_gt, lab_test_predictions, 'finetuned', 'test'))
lab_ap_table.to_csv(lab_dir / 'box_mask_ap_by_size.csv', index=False)
for split, ground, predictions in [('val',lab_val_gt,lab_val_predictions),('test',lab_test_gt,lab_test_predictions)]:
    lab_save(lab_dir / f'coco_{split}_reference.json', ground)
    lab_save(lab_dir / f'coco_{split}_predictions.json', predictions)
lab_original_metrics = pd.read_csv(lab_export / 'metrics_summary.csv').set_index('split')
lab_repeat_checks = []
for split in ('val','test'):
    new_ap = float(lab_ap_table[(lab_ap_table['split']==split) & (lab_ap_table['task']=='segm')]['AP'].iloc[0])
    old_ap = float(lab_original_metrics.loc[split,'mask_map50_95'])
    lab_repeat_checks.append({'split':split,'original_mask_ap':old_ap,'supplement_mask_ap':new_ap,
                              'absolute_difference':abs(new_ap-old_ap)})
lab_save(lab_dir / 'fixed_model_recheck.json', {'comparisons':lab_repeat_checks,
    'scope':'Post-hoc reporting of actual detector box AP and size breakdown. Original test numbers are retained, not replaced.'})
if max(r['absolute_difference'] for r in lab_repeat_checks) > 1e-5:
    raise RuntimeError('The repeated fixed-model mask AP differs from the original. Review the environment before continuing; original results were not changed.')
display(lab_ap_table.round(4))
print('AP_small/medium/large use reference region area at 640x640. Empty bins remain undefined (NaN).')


## 24. Initialization and Fine-Tuned Model

We will reconstruct a one-class pretrained initialization and compare it with the selected model on the same 100 validation frames. We will also use deliberately empty masks to check that the evaluator detects a mask failure even when boxes stay unchanged. No additional training is performed.


In [ ]:
# Reconstruct a one-class pretrained initialization for a validation-only comparison.
# This is not a historical checkpoint: the original initial weights were not saved.
from ultralytics.nn.tasks import SegmentationModel
from ultralytics.utils.torch_utils import init_seeds
lab_initial_path = lab_dir / 'reconstructed_initialization.pt'
lab_initial_meta_path = lab_dir / 'initialization_details.json'
if not lab_initial_path.exists():
    lab_pretrained = YOLO(lab_protocol['model'])
    init_seeds(lab_protocol['seed'], deterministic=True)
    lab_initialized = SegmentationModel(copy.deepcopy(lab_pretrained.model.yaml), nc=1, ch=3, verbose=False)
    lab_initialized.names = {0:'pothole'}
    lab_initialized.load(lab_pretrained.model, verbose=False)
    lab_source_state = lab_pretrained.model.state_dict()
    lab_transfer = []
    for name, tensor in lab_initialized.state_dict().items():
        original = lab_source_state.get(name)
        transferred = original is not None and tuple(original.shape)==tuple(tensor.shape) and torch.equal(original.cpu(),tensor.cpu())
        lab_transfer.append({'tensor':name,'elements':tensor.numel(),'exactly_matches_pretrained':transferred})
    pd.DataFrame(lab_transfer).to_csv(lab_dir / 'weight_transfer.csv', index=False)
    lab_initial_wrapper = YOLO(lab_protocol['model'])
    lab_initial_wrapper.model = lab_initialized
    lab_initial_wrapper.task = 'segment'
    lab_initial_wrapper.save(str(lab_initial_path))
    lab_save(lab_initial_meta_path, {'description':'Reconstructed one-class pretrained initialization, not a saved pre-training checkpoint.',
        'seed':lab_protocol['seed'],'pretrained_source':lab_protocol['model'],
        'pretrained_sha256':lab_sha(Path(lab_pretrained.ckpt_path)),
        'checkpoint_sha256':lab_sha(lab_initial_path),
        'parameters':sum(p.numel() for p in lab_initialized.parameters()),
        'transfer':'Shared detector and mask weights are transferred; the pothole classifier is newly adapted. This differs from the lab Mask R-CNN new-mask-head setup.'})
    del lab_initial_wrapper, lab_initialized, lab_pretrained
    gc.collect()
    torch.cuda.empty_cache()
if lab_sha(lab_initial_path) != lab_read(lab_initial_meta_path)['checkpoint_sha256']:
    raise RuntimeError('Reconstructed baseline checkpoint has changed.')
lab_subset_gt = lab_gt(lab_subset)
lab_initial_predictions, _ = lab_predictions(lab_initial_path, lab_subset, 'initialization_subset')
lab_subset_predictions, lab_subset_times = lab_predictions(lab_checkpoint, lab_subset, 'finetuned_subset')
lab_baseline_table = pd.DataFrame(lab_ap_rows(lab_subset_gt,lab_initial_predictions,'reconstructed_initialization','val100') +
                                lab_ap_rows(lab_subset_gt,lab_subset_predictions,'finetuned','val100'))
lab_baseline_table.to_csv(lab_dir / 'initialization_vs_finetuned.csv',index=False)
lab_save(lab_dir / 'coco_val100_reference.json',lab_subset_gt)
lab_save(lab_dir / 'coco_val100_finetuned.json',lab_subset_predictions)
lab_save(lab_dir / 'coco_val100_initialization.json',lab_initial_predictions)
# A corrupted-mask control tests that box quality cannot hide a mask failure.
lab_bad_masks = [{**p,'segmentation':lab_rle(np.zeros((640,640),bool))} for p in lab_subset_predictions]
lab_diagnostic_table = pd.DataFrame(lab_ap_rows(lab_subset_gt,lab_subset_predictions,'finetuned','diagnostic') +
                                  lab_ap_rows(lab_subset_gt,lab_bad_masks,'empty_mask_control','diagnostic'))
lab_diagnostic_table.to_csv(lab_dir / 'mask_failure_sanity_check.csv',index=False)
display(lab_baseline_table.round(4))
display(lab_diagnostic_table.round(4))
print('The empty-mask control is synthetic. It is not the lab Run B training experiment.')


## 25. Object-Level Precision, Recall and F1

We will plot the validation confidence sweep and report object-level metrics at our original operating point. The diagnostic F1 optimum will be recorded without changing the threshold already selected by Dice. We will also compare manual AP50 with COCOeval.


In [ ]:
# Evaluate object-level P/R/F1 on validation without replacing the locked threshold.
lab_threshold_table = pd.DataFrame([lab_object_scores(lab_val_gt,lab_val_predictions,t)
                                   for t in lab_protocol['threshold_grid']])
lab_threshold_table.to_csv(lab_dir/'validation_object_thresholds.csv',index=False)
lab_best_f1 = max(lab_threshold_table.to_dict('records'),key=lambda r:(r['f1'],r['threshold']))
lab_fixed_val = lab_object_scores(lab_val_gt,lab_val_predictions,lab_threshold)
lab_fixed_test = lab_object_scores(lab_test_gt,lab_test_predictions,lab_threshold)
lab_manual = lab_manual_ap50(lab_subset_gt,lab_subset_predictions)
lab_official, _ = lab_coco_ap(lab_subset_gt,lab_subset_predictions,'segm')
if not np.isclose(lab_manual, lab_official['AP50'], atol=1e-8):
    raise RuntimeError('Manual AP50 does not match COCOeval; inspect matching before reporting.')
lab_save(lab_dir/'object_operating_point.json',{'original_threshold':lab_threshold,
    'original_selection_metric':'validation foreground Dice',
    'validation_f1_optimum_diagnostic_only':lab_best_f1,
    'validation_at_original_threshold':lab_fixed_val,'test_at_original_threshold':lab_fixed_test,
    'manual_ap50':lab_manual,'official_subset_ap50':lab_official['AP50'],
    'note':'This additional F1 analysis was added after original testing. Its optimum does not replace the original threshold; no test threshold sweep is run.'})
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name in ('precision','recall','f1'):
    axes[0].plot(lab_threshold_table.threshold,lab_threshold_table[name],marker='o',label=name)
axes[0].axvline(lab_threshold,color='black',linestyle='--',label='Original operating threshold')
axes[0].set(xlabel='Confidence threshold (validation)',ylabel='Object-level score',ylim=(0,1.02))
axes[0].legend()
axes[1].plot(lab_threshold_table.recall,lab_threshold_table.precision,'o-')
axes[1].set(xlabel='Recall',ylabel='Precision',xlim=(0,1.02),ylim=(0,1.02),title='Validation threshold sweep')
fig.tight_layout()
fig.savefig(lab_dir/'validation_object_thresholds.png',dpi=130)
plt.show()
display(lab_threshold_table.round(4))
print('Diagnostic best F1 threshold:',lab_best_f1['threshold'])
print('Unchanged deployed threshold:',lab_threshold)
print('Manual / COCO AP50:',lab_manual,lab_official['AP50'])


## 26. SAM Prompt and Model Comparisons

We will compare a point prompt with a reference-box prompt, then compare YOLO and SAM on 100 fixed validation images using predicted boxes only. We will record AP, paired mask IoU by size, model size, timing and example overlays. Progress appears every ten images.


In [ ]:
from ultralytics.models.sam import Predictor as LabSAMPredictor

def lab_make_sam():
    return LabSAMPredictor(overrides={'model':lab_protocol['sam_model'], 'task':'segment',
        'mode':'predict','imgsz':1024,'device':lab_device,'verbose':False,'save':False})

def lab_sam_prompts(predictor, shape, boxes=None, points=None, labels=None, multiple=False):
    # Reuse one image embedding; keep every prompt result, regardless of SAM's quality score.
    with torch.inference_mode():
        masks, boxes_scores = predictor.inference_features(predictor.features, src_shape=shape,
            bboxes=boxes, points=points, labels=labels, multimask_output=multiple)
    if masks is None:
        return [], []
    return masks.cpu().numpy().astype(bool), boxes_scores[:,4].cpu().numpy().astype(float)

lab_sam_details = lab_dir / 'sam_details.json'
lab_sam = lab_make_sam()
lab_sam.setup_model()
lab_sam_parameters = sum(p.numel() for p in lab_sam.model.parameters())

# One interior click can produce three interpretations; a reference box is more explicit.
lab_point_path = lab_dir / 'sam_point_vs_box.json'
if not lab_point_path.exists():
    lab_first_image = lab_subset_gt['images'][0]['id']
    lab_object = max((g for g in lab_subset_gt['annotations'] if g['image_id']==lab_first_image),key=lambda g:g['area'])
    lab_reference = lab_decode(lab_object['segmentation'])
    lab_distance = cv2.distanceTransform(lab_reference.astype(np.uint8),cv2.DIST_L2,5)
    lab_y,lab_x = np.unravel_index(np.argmax(lab_distance),lab_distance.shape)
    lab_x0,lab_y0,lab_w,lab_h = lab_object['bbox']
    lab_img = cv2.imread(str(lab_frame_dir/lab_subset[0]['image_path']))
    lab_sam.set_image(lab_img)
    lab_points,lab_quality = lab_sam_prompts(lab_sam,lab_reference.shape,points=[[int(lab_x),int(lab_y)]],labels=[1],multiple=True)
    lab_boxes,lab_box_quality = lab_sam_prompts(lab_sam,lab_reference.shape,boxes=[[lab_x0,lab_y0,lab_x0+lab_w,lab_y0+lab_h]])
    if len(lab_points)!=3 or len(lab_boxes)!=1:
        raise RuntimeError('Expected three point-prompt masks and one box-prompt mask.')
    lab_point_scores = [lab_iou(m,lab_reference) for m in lab_points]
    lab_box_score = lab_iou(lab_boxes[0],lab_reference)
    fig,axes=plt.subplots(1,6,figsize=(18,3))
    axes[0].imshow(cv2.cvtColor(lab_img,cv2.COLOR_BGR2RGB))
    axes[0].scatter([lab_x],[lab_y],c='red',s=25)
    axes[0].set_title('Validation image + click')
    for ax,mask,title in zip(axes[1:],[lab_reference,*lab_points,lab_boxes[0]],
        ['Reference']+[f'Point {i+1}: predicted IoU {q:.3f}\nMeasured IoU {s:.3f}' for i,(q,s) in enumerate(zip(lab_quality,lab_point_scores))]+[f'Box: measured IoU {lab_box_score:.3f}']):
        ax.imshow(mask,cmap='gray');ax.set_title(title)
    for ax in axes: ax.axis('off')
    fig.tight_layout();fig.savefig(lab_dir/'sam_point_vs_box.png',dpi=130,bbox_inches='tight');plt.close(fig)
    lab_save(lab_point_path,{'video_id':lab_subset[0]['video_id'],'frame_index':26,
        'prompt_source':'Validation reference mask; assisted prompt comparison, not automatic model performance.',
        'point_xy':[int(lab_x),int(lab_y)],'point_predicted_quality':list(map(float,lab_quality)),
        'point_measured_iou':lab_point_scores,'box_measured_iou':lab_box_score})

lab_by_image=defaultdict(list)
for lab_p in lab_subset_predictions:
    if lab_p['score']>=lab_threshold:
        lab_by_image[lab_p['image_id']].append(lab_p)
lab_sam_predictions,lab_sam_times,lab_sam_devices=[],[],[]
lab_sam_cache=lab_dir/'sam_prediction_cache'
lab_sam_cache.mkdir(exist_ok=True)
# Warm-up is excluded from timing; per-photo timing includes the encoder and mask decoding.
lab_sam.set_image(np.zeros((640,640,3),np.uint8))
lab_sam_prompts(lab_sam,(640,640),boxes=[[100,100,400,400]])
for lab_image_id,lab_row in enumerate(lab_subset,1):
    lab_kept=lab_by_image[lab_image_id]
    lab_path=lab_sam_cache/f"{lab_row['video_id']}_f26.json"
    lab_identity=lab_hash({'image':lab_row['image_sha256'],'detections':lab_kept,'sam':lab_protocol['sam_model'],'spec':lab_spec})
    if lab_path.exists():
        lab_saved=lab_read(lab_path)
        if lab_saved['fingerprint']!=lab_identity:
            raise RuntimeError('Existing SAM comparison uses different prompts/settings.')
    else:
        lab_img=cv2.imread(str(lab_frame_dir/lab_row['image_path']))
        lab_boxes_xyxy=[[p['bbox'][0],p['bbox'][1],p['bbox'][0]+p['bbox'][2],p['bbox'][1]+p['bbox'][3]] for p in lab_kept]
        lab_sync();lab_start=time.perf_counter()
        lab_instances=[]
        if lab_kept:
            lab_sam.set_image(lab_img)
            lab_masks,_=lab_sam_prompts(lab_sam,lab_img.shape[:2],boxes=lab_boxes_xyxy)
            if len(lab_masks)!=len(lab_kept):
                raise RuntimeError('SAM must return exactly one mask for each kept detector box.')
            lab_instances=[{**p,'segmentation':lab_rle(mask)} for p,mask in zip(lab_kept,lab_masks)]
        lab_sync()
        lab_saved={'fingerprint':lab_identity,'seconds':time.perf_counter()-lab_start,'instances':lab_instances,
                   'timing_device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
                   'torch_version':str(torch.__version__)}
        lab_save(lab_path,lab_saved)
    lab_sam_predictions.extend(lab_saved['instances'])
    lab_sam_times.append(lab_saved['seconds'])
    lab_sam_devices.append(lab_saved.get('timing_device','not recorded'))
    if lab_image_id==1 or lab_image_id%10==0:
        print(f'SAM comparison: {lab_image_id}/100 validation videos',flush=True)
lab_kept_predictions=[p for p in lab_subset_predictions if p['score']>=lab_threshold]
assert len(lab_sam_predictions)==len(lab_kept_predictions)
lab_sam_ap,_=lab_coco_ap(lab_subset_gt,lab_sam_predictions,'segm')
lab_yolo_matched_ap,_=lab_coco_ap(lab_subset_gt,lab_kept_predictions,'segm')
lab_sam_table=pd.DataFrame([{'method':'YOLO masks (kept detections)',**lab_yolo_matched_ap},
                           {'method':'SAM masks (same detections)',**lab_sam_ap}])
lab_sam_table.to_csv(lab_dir/'sam_vs_model_ap.csv',index=False)
lab_save(lab_dir/'coco_val100_sam.json',lab_sam_predictions)

# Pair GT objects one-to-one to detector boxes; reuse that same pairing for both masks.
lab_paired=[]
lab_sam_by_image=defaultdict(list)
lab_gt_by_image=defaultdict(list)
for p in lab_sam_predictions: lab_sam_by_image[p['image_id']].append(p)
for g in lab_subset_gt['annotations']: lab_gt_by_image[g['image_id']].append(g)
for image_id,objects in lab_gt_by_image.items():
    kept=lab_by_image[image_id]
    sam_items=lab_sam_by_image[image_id]
    if not kept: continue
    matrix=lab_mask_util.iou([p['bbox'] for p in kept],[g['bbox'] for g in objects],[0]*len(objects))
    used=set()
    for i in sorted(range(len(kept)),key=lambda j:-kept[j]['score']):
        candidates=[(float(matrix[i,j]),j) for j,g in enumerate(objects) if j not in used and g['category_id']==kept[i]['category_id']]
        overlap,j=max(candidates,default=(-1.,-1))
        if overlap<.5: continue
        used.add(j)
        reference=lab_decode(objects[j]['segmentation'])
        lab_paired.append({'image_id':image_id,'gt_id':objects[j]['id'],'size':lab_size(objects[j]['area']),
            'box_iou':overlap,'model_iou':lab_iou(lab_decode(kept[i]['segmentation']),reference),
            'sam_iou':lab_iou(lab_decode(sam_items[i]['segmentation']),reference)})
lab_paired_table=pd.DataFrame(lab_paired,columns=['image_id','gt_id','size','box_iou','model_iou','sam_iou'])
lab_paired_table.to_csv(lab_dir/'sam_paired_objects.csv',index=False)
lab_sizes=[]
for size in ('small','medium','large'):
    part=lab_paired_table[lab_paired_table['size']==size]
    lab_sizes.append({'size':size,'matched_objects':len(part),
        'model_median_iou':float(part.model_iou.median()) if len(part) else None,
        'sam_median_iou':float(part.sam_iou.median()) if len(part) else None,
        'sam_better_fraction':float((part.sam_iou>part.model_iou).mean()) if len(part) else None})
pd.DataFrame(lab_sizes).to_csv(lab_dir/'sam_paired_by_size.csv',index=False)
lab_saved_model=YOLO(str(lab_checkpoint))
lab_model_parameters=sum(p.numel() for p in lab_saved_model.model.parameters())
del lab_saved_model,lab_sam
gc.collect();torch.cuda.empty_cache()
lab_save(lab_sam_details,{'images':100,'distinct_videos':100,'detections':len(lab_kept_predictions),
    'sam_additional_ms_per_image':1000*float(np.mean(lab_sam_times)),
    'model_pipeline_ms_per_image':1000*float(np.mean(lab_subset_times)),
    'combined_model_plus_sam_ms_per_image':1000*float(np.mean(lab_sam_times)+np.mean(lab_subset_times)),
    'sam_parameters':lab_sam_parameters,'model_parameters':lab_model_parameters,
    'sam_timing_device_counts':dict(Counter(lab_sam_devices)),
    'timing_scope':'Warm-up excluded; model timing includes image IO/postprocessing/RLE. SAM is additional encoder+prompt/postprocessing/RLE time after image loading. Sequential measurements, not production-service latency.',
    'ap_scope':'Both methods use exactly the same retained detections at the original operating threshold. This conditional comparison is not the unrestricted headline AP.',
    'prompt_source':'Predicted boxes only; classes and confidence scores remain the detector outputs.',
    'matching':'One-to-one score-ordered detector-box matching at box IoU>=0.5; same pairs for model and SAM.'})

# Show the first three predetermined validation images, not selected by performance.
fig,axes=plt.subplots(3,3,figsize=(12,12))
for row_index,image_id in enumerate((1,2,3)):
    row=lab_subset[image_id-1]
    rgb=cv2.cvtColor(cv2.imread(str(lab_frame_dir/row['image_path'])),cv2.COLOR_BGR2RGB)
    groups=[lab_gt_by_image[image_id],lab_by_image[image_id],lab_sam_by_image[image_id]]
    for column,items in enumerate(groups):
        union=np.zeros((640,640),bool)
        for item in items: union|=lab_decode(item['segmentation'])
        overlay=rgb.copy();overlay[union]=(0.55*overlay[union]+0.45*np.array([0,255,0])).astype(np.uint8)
        axes[row_index,column].imshow(overlay)
        axes[row_index,column].set_title(f"Val {row['video_id']} | "+['Reference','YOLO','SAM from predicted boxes'][column])
        axes[row_index,column].axis('off')
fig.tight_layout();fig.savefig(lab_dir/'sam_comparison_gallery.png',dpi=120,bbox_inches='tight');plt.show()
display(Image.open(lab_dir/'sam_point_vs_box.png'))
display(lab_sam_table.round(4));display(pd.DataFrame(lab_sizes).round(4))
print('SAM extra milliseconds per image:',lab_read(lab_sam_details)['sam_additional_ms_per_image'])


## 27. Laboratory Results and Interpretation

We will summarize the measured comparisons and explain their limitations. A task-by-task table will show how the lab was adapted to potholes and which architecture-specific demonstrations were replaced.


In [ ]:
# Summarize the measured results and explain how the lab was adapted.
lab_history=pd.read_csv(lab_export/'training_history.csv')
lab_history.columns=lab_history.columns.str.strip()
fig,axes=plt.subplots(1,2,figsize=(12,4))
for column in ('train/box_loss','train/cls_loss','train/seg_loss'):
    axes[0].plot(lab_history.epoch,lab_history[column].ewm(alpha=.1,adjust=False).mean(),label=column)
for column in ('metrics/mAP50-95(B)','metrics/mAP50-95(M)'):
    axes[1].plot(lab_history.epoch,lab_history[column],label=column)
axes[0].set(title='Saved training losses, EMA alpha=0.1',xlabel='Epoch',ylabel='Loss')
axes[1].set(title='Saved internal validation AP (polygon targets)',xlabel='Epoch',ylabel='AP')
for ax in axes: ax.legend()
fig.tight_layout();fig.savefig(lab_dir/'training_component_curves.png',dpi=130);plt.show()

lab_mapping=[
    (1,'Training annotation summary','data_exercises.json','One pothole class; derived-region counts and size bins; no source crowd labels.'),
    (2,'Polygon simplification','polygon_simplification.csv','Up to 150 training regions per size bin; eps 0,1,2,4 at evaluation resolution.'),
    (3,'RLE encoding/decoding','data_exercises.json','Manual column-major round trip checked against pycocotools; wrong-order IoU recorded.'),
    (4,'Experiment configuration','../protocol.json; extension_protocol.json','Retain the original YOLO recipe and frozen operating threshold.'),
    (5,'Transferred and newly adapted weights','weight_transfer.csv; initialization_details.json','YOLO class adaptation instead of a new Mask R-CNN mask head; initialization reconstructed retrospectively.'),
    (6,'COCO prediction format','coco_val100_reference.json; coco_val100_finetuned.json','Explicit detector boxes, masks, confidence and one pothole category.'),
    (7,'Initialization baseline AP','initialization_vs_finetuned.csv','Same fixed 100 validation images; COCO classes are not mislabeled as potholes.'),
    (8,'Manual mask AP50','object_operating_point.json','Score-ordered one-to-one matching and 101-point interpolation agree with COCOeval.'),
    (9,'Training curves','training_component_curves.png','Original successful training only. Mask R-CNN ln(2) stalled-head training is not repeated for YOLO.'),
    (10,'Box/mask AP overall and by size','box_mask_ap_by_size.csv; initialization_vs_finetuned.csv','Mask AP uses mask areas. COCO size bins refer to 640x640 derived regions.'),
    (11,'Diagnosing a mask-specific failure','mask_failure_sanity_check.csv','Synthetic empty-mask control: same detector boxes, deliberately corrupted masks. Not a second trained run.'),
    (12,'Object-level threshold analysis','validation_object_thresholds.csv','Validation-only PR/F1 diagnostic; does not replace the threshold already chosen by Dice.'),
    (13,'Test metrics, gallery and handoff','box_mask_ap_by_size.csv; ../figures/test_qualitative_gallery.png; ../HANDOFF.md','Frozen checkpoint/threshold; additional test box reporting is explicitly post-hoc; original test results stay primary.'),
    (14,'SAM point versus box','sam_point_vs_box.png; sam_point_vs_box.json','Three point interpretations and one reference-box prompt on a validation pothole.'),
    (15,'SAM from predicted boxes','sam_vs_model_ap.csv; sam_paired_by_size.csv; sam_details.json','100 distinct validation videos; same detections/scores/classes; paired IoU by size and measured timing.'),
    (16,'Comparison gallery','sam_comparison_gallery.png','Three predetermined validation images: reference, YOLO and SAM.')]
lab_map_table=pd.DataFrame(lab_mapping,columns=['lab_task','topic','evidence','application_adaptation'])
lab_map_table.to_csv(lab_dir/'lab_alignment.csv',index=False)

def lab_markdown_table(frame):
    def cell(value):
        if value is None or (isinstance(value,(float,np.floating)) and np.isnan(value)):
            return 'N/A'
        if isinstance(value,(float,np.floating)):
            return f'{value:.4f}'
        return str(value).replace('|','/').replace('\n',' ')
    header='| '+' | '.join(map(str,frame.columns))+' |'
    separator='|'+'|'.join(['---']*len(frame.columns))+'|'
    return '\n'.join([header,separator]+['| '+' | '.join(cell(x) for x in row)+' |' for row in frame.itertuples(index=False,name=None)])

lab_stats=lab_read(lab_sam_details)
lab_point=lab_read(lab_point_path)
lab_operating=lab_read(lab_dir/'object_operating_point.json')
lab_sam_comparison=pd.read_csv(lab_dir/'sam_vs_model_ap.csv')
lab_size_comparison=pd.read_csv(lab_dir/'sam_paired_by_size.csv')
lab_sam_delta=float(lab_sam_comparison.iloc[1]['AP']-lab_sam_comparison.iloc[0]['AP'])
lab_size_text=[]
for row in lab_size_comparison.to_dict('records'):
    if row['matched_objects']:
        lab_size_text.append(f"- {row['size']}: {int(row['matched_objects'])} paired regions; model/SAM median IoU "
            f"{row['model_median_iou']:.4f}/{row['sam_median_iou']:.4f}; SAM wins on {100*row['sam_better_fraction']:.1f}%.")
    else:
        lab_size_text.append(f"- {row['size']}: no matched regions; no conclusion for this size bin.")
lab_report=f'''# M5 laboratory adaptation and additional evaluation

Contributor: Abdulrahman Alsalmi, 225493.

## Scope and chronology
The reference is Day6_Lab_Instance_segmentation_v2_student.ipynb. Its experiments are adapted to potholes and
YOLO11n-seg. The project guide asks for application-adapted laboratory deliverables; this is not a claim that
the COCO/Detectron2 classroom notebook was reproduced verbatim. Architecture-specific substitutions are listed below.
These analyses were added after the original test evaluation. They do not select a new model or operating point.
The original model fingerprint is {lab_selection['checkpoint_sha256']}; its threshold stays {lab_threshold}.
The original Dice/IoU/area results remain the primary benchmark. The additional pass with this fixed model collects
actual detector boxes for box AP and object-level reporting; it is not a fresh test used for model selection.
Reference masks are binary semantics. Every instance ID in this supplement is an 8-connected region, not a native
physical-pothole identity. Crowds are not annotated. All size bins use the 640x640 reference-region area.

## Data representations
Training objects, polygon simplification and the RLE check are recorded in data_exercises.json and
polygon_simplification.csv. External contours can fill holes or lose tiny regions; this exercise does not change
the reference annotations. Column-major manual RLE was verified against pycocotools.

## Initialization and trained model
The original initial checkpoint was not saved. A one-class initialization is reconstructed from the recorded
pretrained YOLO11n-seg source and seed. Shared box/mask weights transfer; the pothole classifier is newly adapted.
This is a retrospective initialization control, not proof of the exact original initial weight values.
It is evaluated only on the same 100 fixed validation frames as the trained model, one frame from each of 100 videos.
The lab's 80-class COCO zoo model is not presented as a pothole classifier.

{lab_markdown_table(pd.read_csv(lab_dir/'initialization_vs_finetuned.csv'))}

Training losses and internal validation AP come from the original 30-epoch history. Those internal AP numbers
use polygon labels. YOLO's mask loss is not the lab's isolated Mask R-CNN BCE, so the ln(2) reference is not imposed.
The empty-mask diagnostic deliberately changes masks while keeping detections fixed. It checks metric behavior;
it is not the lab's second stalled-head training run and must not be described as one.

## Box versus mask AP
{lab_markdown_table(pd.read_csv(lab_dir/'box_mask_ap_by_size.csv'))}

AP uses the original low confidence floor {lab_protocol['eval_conf_floor']}, not the deployment threshold.
Undefined size bins remain N/A. The repeated fixed-model mask AP is compared with the original in fixed_model_recheck.json.

## Object-level precision, recall and F1
One-to-one, class-aware mask matching uses IoU >= 0.5. These are object-level metrics, distinct from pixel Dice.
The diagnostic maximum validation F1 is {lab_best_f1['f1']:.4f} at confidence {lab_best_f1['threshold']}.
The deployed confidence remains {lab_threshold}, selected earlier by validation foreground Dice.
At this original threshold, validation precision/recall/F1 are {lab_fixed_val['precision']:.4f}/
{lab_fixed_val['recall']:.4f}/{lab_fixed_val['f1']:.4f}; test values are {lab_fixed_test['precision']:.4f}/
{lab_fixed_test['recall']:.4f}/{lab_fixed_test['f1']:.4f}. The test threshold is not swept or changed.
Manual mask AP50 ({lab_manual:.6f}) agrees with COCOeval ({lab_official['AP50']:.6f}) on the 100-frame subset.

## SAM prompts and detector-box comparison
The one-click exercise produces three masks with measured reference IoU {lab_point['point_measured_iou']}.
The reference-box prompt reaches {lab_point['box_measured_iou']:.4f}. This prompted exercise uses annotation information
and is not automatic test performance.

The main SAM comparison instead uses predicted boxes only. It retains the detector's classes, scores and detection
count. Both methods are evaluated on the same 100 validation images at the original retained-detection threshold.
This conditional AP must not replace the unrestricted model headline AP.

{lab_markdown_table(lab_sam_comparison)}

SAM minus YOLO mask AP on these same detections: {lab_sam_delta:+.4f}.
{chr(10).join(lab_size_text)}

SAM has {lab_stats['sam_parameters']:,} parameters; YOLO has {lab_stats['model_parameters']:,}.
Measured YOLO pipeline time is {lab_stats['model_pipeline_ms_per_image']:.2f} ms/image, with SAM adding
{lab_stats['sam_additional_ms_per_image']:.2f} ms/image. Their sequential sum is
{lab_stats['combined_model_plus_sam_ms_per_image']:.2f} ms/image. Warm-up is excluded; model timing includes
image loading and mask serialization, whereas SAM timing begins after image loading. These are local comparative
measurements, not HTTP service latency or a production benchmark. Cached timings may come from an earlier session;
device counts are saved in finetuned_subset_timing.json and sam_details.json.
SAM still inherits missed objects, false detections and class/score errors from its prompts. This analysis does not
replace the selected deployable model. SAM remains an annotation aid unless a separate validation-only deployment
experiment justifies its added cost. Three predetermined comparisons are saved in sam_comparison_gallery.png.

## Laboratory-to-project mapping
{lab_markdown_table(lab_map_table)}

## References
- Project guidelines, M5 rubric and road-damage project requirements, pages 6-7 and 32-33.
- Course handout: Day6_Lab_Instance_segmentation_v2_student.ipynb.
- SAM predictor API: https://docs.ultralytics.com/reference/models/sam/predict/
- Segmentation model initialization: https://docs.ultralytics.com/reference/models/yolo/segment/train/
'''
(lab_dir/'M5_Lab_Supplement.md').write_text(lab_report,encoding='utf-8')
for name,digest in lab_original_hashes.items():
    if lab_sha(lab_export/name)!=digest:
        raise RuntimeError(f'Original artifact changed: {name}')
lab_save(lab_dir/'supplement_complete.json',{'status':'COMPLETE','scope':'Application-adapted lab analyses; architecture-specific substitutions are disclosed.',
    'original_results_preserved':True,'original_hashes':lab_original_hashes,
    'protocol':lab_hash(lab_spec),'sam_comparison_images':100})
display(lab_map_table)
print('Supplement report saved:',lab_dir/'M5_Lab_Supplement.md')
print('Original checkpoint, threshold, selection and primary metric files are unchanged.')


## 28. Tracking the Additional Analyses

We will upload the supplementary tables and evidence to a separate W&B run using a fresh connection. The original model experiment remains available through its saved link.


In [ ]:
# Upload saved results with visible progress.
# Training and evaluation are not repeated.

import json
import netrc
import os
import re
import signal
import subprocess
import sys
import tempfile
import time
import uuid

from pathlib import Path
from getpass import getpass


lab_worker = r'''
import json
import os
from pathlib import Path

root = Path(__file__).resolve().parent
p = json.loads((root / "payload.json").read_text(encoding="utf-8"))


def phase(message):
    temp = root / "phase.part"
    temp.write_text(message, encoding="utf-8")
    temp.replace(root / "phase.txt")


phase("Preparing saved files from Drive")

source = Path(p["source"])

files = [
    f for f in sorted(source.iterdir())
    if f.is_file()
    and f.suffix.lower() in (".csv", ".json", ".png", ".md", ".txt", ".pt")
    and f.name not in ("wandb_supplement_tracking.json", "wandb_upload.log")
]

required = [
    "box_mask_ap_by_size.csv",
    "initialization_vs_finetuned.csv",
    "validation_object_thresholds.csv",
    "sam_vs_model_ap.csv",
    "sam_paired_by_size.csv",
    "mask_failure_sanity_check.csv",
    "sam_details.json",
    "supplement_complete.json",
    "extension_protocol.json",
]

missing = set(required) - {f.name for f in files}

if missing:
    raise RuntimeError(
        "Missing saved supplement files: " + ", ".join(sorted(missing))
    )

# Copying is included in the monitored timeout.
for index, file in enumerate(files, 1):
    phase(f"Copying file {index}/{len(files)}: {file.name}")

    size = file.stat().st_size
    copied = 0

    with file.open("rb") as src, (root / file.name).open("wb") as dst:
        while True:
            block = src.read(4 * 1024**2)

            if not block:
                break

            dst.write(block)
            copied += len(block)

            phase(
                f"Copying file {index}/{len(files)}: {file.name} "
                f"({copied / 1024**2:.1f}/{size / 1024**2:.1f} MiB)"
            )

complete = json.loads(
    (root / "supplement_complete.json").read_text(encoding="utf-8")
)

if complete.get("status") != "COMPLETE":
    raise RuntimeError(
        "Section 27 has not completed. Original results are unchanged."
    )

phase("Loading W&B libraries")

import pandas as pd
import requests
import wandb

phase("Checking account and project access")

response = requests.post(
    "https://api.wandb.ai/graphql",
    auth=("api", os.environ["WANDB_API_KEY"]),
    json={
        "query": (
            "query($e:String!,$p:String!){"
            "viewer{username} "
            "project(name:$p,entityName:$e){name}}"
        ),
        "variables": {
            "e": p["entity"],
            "p": p["project"],
        },
    },
    timeout=(10, 20),
    allow_redirects=False,
)

if response.status_code != 200:
    raise RuntimeError(
        f"W&B access check returned HTTP {response.status_code}."
    )

reply = response.json()
data = reply.get("data") or {}

if (
    reply.get("errors")
    or not data.get("viewer")
    or not data.get("project")
):
    raise RuntimeError(
        "The account cannot access the selected W&B project."
    )

phase("Starting W&B run: maximum initialization wait is 60 seconds")

run = wandb.init(
    entity=p["entity"],
    project=p["project"],
    name=p["name"],
    id=p["run_id"],
    resume="allow",
    mode="online",
    job_type="supplementary-evaluation",
    tags=["M5", "lab-adaptation", "validation-comparison"],
    config=json.loads(
        (root / "extension_protocol.json").read_text(encoding="utf-8")
    ),
    dir=str(root / "logs"),
    settings=wandb.Settings(init_timeout=60),
)

old_tracking = Path(p["original_tracking"])

if old_tracking.is_file():
    old_info = json.loads(old_tracking.read_text(encoding="utf-8"))
    run.summary["original_model_run"] = old_info.get("url")

run.summary["original_model_and_threshold_changed"] = False

phase("Uploading the six result tables")

for name in required[:6]:
    run.log({
        Path(name).stem: wandb.Table(
            dataframe=pd.read_csv(root / name)
        )
    })

details = json.loads(
    (root / "sam_details.json").read_text(encoding="utf-8")
)

for key, value in details.items():
    if isinstance(value, (int, float)):
        run.summary["supplement/" + key] = value

phase("Uploading figures, reports and evaluation files")

artifact = wandb.Artifact(
    p["name"],
    type="evaluation",
)

for file in files:
    artifact.add_file(
        str(root / file.name),
        name=file.name,
    )

run.log_artifact(artifact).wait(timeout=180)

url = run.url

phase("Confirming upload completion")
run.finish()

(root / "result.json").write_text(
    json.dumps({
        "status": "SYNCED",
        "run_id": p["run_id"],
        "url": url,
    }),
    encoding="utf-8",
)

phase("Upload completed")
'''


def lab_stop_upload(process):
    """Stop only the upload process created by this cell."""
    if process.poll() is not None:
        return

    try:
        if os.name == "posix":
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()

        process.wait(timeout=5)

    except subprocess.TimeoutExpired:
        if os.name == "posix":
            os.killpg(process.pid, signal.SIGKILL)
        else:
            process.kill()

        process.wait(timeout=5)

    except ProcessLookupError:
        pass


def lab_upload_saved_supplement():
    global lab_dir, lab_tracking_path

    print(
        "Section 28: uploading saved results only.",
        flush=True,
    )

    # Recover the saved folder if the notebook variables are unavailable.
    if "lab_dir" not in globals():
        from google.colab import drive

        drive.mount("/content/drive")

        projects = [
            p for p in Path("/content/drive/MyDrive").iterdir()
            if p.is_dir() and p.name.strip() == "AI Project"
        ]

        if len(projects) != 1:
            raise RuntimeError(
                "Cannot locate AI Project in MyDrive."
            )

        lab_dir = (
            projects[0]
            / "M5_Segmentation"
            / "runs"
            / "m5_yolo11n_seg_v1"
            / "lab_extension_v1"
        )

    source = Path(lab_dir)
    lab_tracking_path = source / "wandb_supplement_tracking.json"

    print("Reading the previous upload status...", flush=True)

    previous = (
        json.loads(lab_tracking_path.read_text(encoding="utf-8"))
        if lab_tracking_path.exists()
        else {}
    )

    run_id = previous.get("run_id") or uuid.uuid4().hex[:8]

    # Read the saved key without starting W&B in the notebook process.
    key = os.environ.get("WANDB_API_KEY", "").strip()

    if not key:
        try:
            auth = netrc.netrc().authenticators("api.wandb.ai")
            key = auth[2] if auth else ""

        except (OSError, netrc.NetrcParseError):
            pass

    if not key:
        key = getpass(
            "Paste W&B API key, then press Enter: "
        ).strip()

    if not key:
        print(
            "No key entered. Upload remains pending; "
            "saved results are unchanged."
        )
        return

    print(
        "Key received. Preparing the upload process...",
        flush=True,
    )

    root = Path(tempfile.mkdtemp(prefix="m5_lab_upload_"))

    # Start a fresh connection, separate from earlier W&B attempts.
    env = {
        k: v for k, v in os.environ.items()
        if not k.startswith(("WANDB_", "_WANDB_"))
    }

    env.update(
        WANDB_API_KEY=key,
        WANDB_BASE_URL="https://api.wandb.ai",
        WANDB_CONSOLE="off",
    )

    for variable, folder in [
        ("WANDB_DIR", "logs"),
        ("WANDB_CACHE_DIR", "cache"),
        ("WANDB_DATA_DIR", "data"),
        ("WANDB_CONFIG_DIR", "config"),
    ]:
        (root / folder).mkdir()
        env[variable] = str(root / folder)

    payload = {
        "run_id": run_id,
        "entity": "alharby6626-a",
        "project": "rdd2022-road-damage",
        "name": "m5_yolo11n_seg_v1-lab-supplement",
        "source": str(source),
        "original_tracking": str(
            source.parents[2]
            / "exports"
            / "m5_yolo11n_seg_v1"
            / "wandb_tracking.json"
        ),
    }

    (root / "payload.json").write_text(
        json.dumps(payload),
        encoding="utf-8",
    )

    (root / "upload.py").write_text(
        lab_worker,
        encoding="utf-8",
    )

    info = {
        "run_id": run_id,
        "status": "PENDING UPLOAD",
        "local_directory": str(root),
    }

    process = None
    interrupted = False
    error = ""
    started = time.monotonic()

    print(
        "Starting. Progress will appear every 10 seconds; "
        "maximum wait is 5 minutes.",
        flush=True,
    )

    try:
        with (root / "process.log").open(
            "w", encoding="utf-8"
        ) as output:

            process = subprocess.Popen(
                [sys.executable, "-u", str(root / "upload.py")],
                cwd=root,
                env=env,
                stdout=output,
                stderr=subprocess.STDOUT,
                start_new_session=(os.name == "posix"),
            )

            while True:
                try:
                    process.wait(timeout=10)
                    break

                except subprocess.TimeoutExpired:
                    stage_path = root / "phase.txt"

                    stage = (
                        stage_path.read_text(encoding="utf-8")
                        if stage_path.exists()
                        else "Starting process"
                    )

                    elapsed = int(time.monotonic() - started)

                    print(
                        f"{elapsed}s: {stage}",
                        flush=True,
                    )

                    if elapsed >= 300:
                        raise TimeoutError(
                            "Upload stopped after 5 minutes at: "
                            + stage
                        )

        result = root / "result.json"

        if process.returncode == 0 and result.exists():
            info = json.loads(
                result.read_text(encoding="utf-8")
            )
        else:
            error = "The upload process returned an error."

    except KeyboardInterrupt:
        interrupted = True
        error = "Upload interrupted by the user."

    except Exception as exc:
        error = str(exc)

    finally:
        if process is not None:
            lab_stop_upload(process)

        env.pop("WANDB_API_KEY", None)

    log_path = root / "process.log"

    log = (
        log_path.read_text(encoding="utf-8", errors="replace")
        if log_path.exists()
        else ""
    )

    log = log.replace(key, "[REDACTED]")
    log = re.sub(r"wandb_v1_\S+", "[REDACTED]", log)

    del key

    if error:
        info["message"] = error

    print("Saving upload status to Drive...", flush=True)

    lab_tracking_path.write_text(
        json.dumps(info, indent=2),
        encoding="utf-8",
    )

    (source / "wandb_upload.log").write_text(
        log,
        encoding="utf-8",
    )

    if info["status"] == "SYNCED":
        print("SUCCESS - W&B run:", info["url"])
        print(
            "Run Section 29 next to create the updated ZIP."
        )

    else:
        print(error)
        print("\n".join(log.splitlines()[-10:]))

        print(
            "W&B upload is still PENDING. "
            "Your saved calculations and results are unchanged."
        )

        print(
            "You may run Section 29 to save the package. "
            "Retry only Section 28 later, then rerun Section 29."
        )

    if interrupted:
        raise KeyboardInterrupt(
            "Upload stopped; results preserved."
        )


lab_upload_saved_supplement()

## 29. Saving the Updated Submission

We will create a new submission ZIP containing the original results and the laboratory supplement. We will retain the original package and save this notebook with its new outputs after the remaining sections finish.


In [ ]:
# Build a separate complete package, leaving the previous submission ZIP intact.
for name,digest in lab_original_hashes.items():
    if lab_sha(lab_export/name)!=digest:
        raise RuntimeError(f'Original artifact changed: {name}')
if lab_read(lab_dir/'supplement_complete.json')['status']!='COMPLETE':
    raise RuntimeError('Complete the additional analyses before packaging.')
lab_final=lab_m5/'exports'/f'{lab_experiment}_with_lab'
shutil.copytree(lab_export,lab_final,dirs_exist_ok=True)
shutil.copytree(lab_dir,lab_final/'lab_extension',dirs_exist_ok=True,
               ignore=shutil.ignore_patterns('prediction_cache','sam_prediction_cache','wandb_upload.log'))
lab_old_report=(lab_export/'M5_Report.md').read_text(encoding='utf-8')
(lab_final/'M5_Report_before_lab.md').write_text(lab_old_report,encoding='utf-8')
lab_updated_report=lab_old_report.replace(
    'The exact M5 lab handout was unavailable; additional lab-specific deliverables still need a cross-check.',
    'The Day 6 instance-segmentation lab has now been reviewed. Application-adapted analyses and disclosed architecture-specific substitutions are documented in lab_extension/M5_Lab_Supplement.md.')
lab_updated_report+='\n## Additional laboratory analyses\nSee lab_extension/M5_Lab_Supplement.md and lab_extension/lab_alignment.csv. The original checkpoint, threshold and primary test metrics are unchanged.\n'
lab_tracking=lab_read(lab_tracking_path) if lab_tracking_path.exists() else {'status':'NOT UPLOADED'}
if lab_tracking.get('status')=='SYNCED':
    lab_updated_report+='Supplementary W&B run: '+lab_tracking['url']+'\n'
else:
    lab_updated_report+='Supplementary W&B upload remains pending; local analyses are included.\n'
(lab_final/'M5_Report.md').write_text(lab_updated_report,encoding='utf-8')
lab_checklist=pd.read_csv(lab_export/'completion_checklist.csv').fillna('')
lab_row=lab_checklist['requirement'].str.startswith('Cross-check any additional M5')
lab_checklist.loc[lab_row,'status']='REVIEWED AND ADAPTED'
lab_checklist.loc[lab_row,'evidence']='lab_extension/lab_alignment.csv; architecture-specific substitutions explicitly documented'
lab_checklist=pd.concat([lab_checklist,pd.DataFrame([{'requirement':'Additional lab comparisons and analysis',
    'status':'COMPLETE','evidence':'lab_extension/M5_Lab_Supplement.md'},
    {'requirement':'Supplementary W&B tracking','status':'COMPLETE' if lab_tracking.get('status')=='SYNCED' else 'PENDING UPLOAD',
     'evidence':lab_tracking.get('url','lab_extension/wandb_supplement_tracking.json')}])],ignore_index=True)
lab_checklist.to_csv(lab_final/'completion_checklist.csv',index=False)
lab_inventory=[{'file':p.relative_to(lab_final).as_posix(),'bytes':p.stat().st_size,'sha256':lab_sha(p)}
               for p in sorted(lab_final.rglob('*')) if p.is_file() and p.name!='artifact_manifest.json']
lab_save(lab_final/'artifact_manifest.json',lab_inventory)
lab_package=lab_m5/'exports'/f'{lab_experiment}_submission_with_lab.zip'
lab_temporary=lab_package.with_suffix('.zip.part')
with zipfile.ZipFile(lab_temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(lab_final.rglob('*')):
        if path.is_file(): archive.write(path,(Path(lab_experiment)/path.relative_to(lab_final)).as_posix())
lab_temporary.replace(lab_package)
with zipfile.ZipFile(lab_package) as archive:
    if archive.testzip() is not None: raise RuntimeError('ZIP verification failed.')
print('Updated package:',lab_package)
print('Size (MiB):',round(lab_package.stat().st_size/1024**2,1))
print('Original model and primary results preserved. Original submission ZIP retained.')
print('Save this notebook with its new outputs, then download it as M5_Pothole_Segmentation.ipynb.')
